In [ ]:
# @title
# ============================================================
# GEMMA 4 E4B-it + MTP ASSISTANT OPTIMIZED TEST FOR COLAB T4
# Một cell duy nhất:
# cài -> login HF -> load E4B target -> load assistant -> benchmark baseline vs MTP -> test DFUZZ JSON
# ============================================================

import os, sys, gc, re, json, time, subprocess, traceback
from importlib.metadata import PackageNotFoundError, version

print("=" * 110)
print("0) CÀI / CẬP NHẬT THƯ VIỆN")
print("=" * 110)

PIP_PACKAGES = [
    "transformers>=5.5.0",
    "accelerate>=1.8.0",
    "bitsandbytes>=0.49.0",
    "huggingface_hub>=0.36.0",
    "sentencepiece",
    "protobuf<6,>=5.29.1",
]

def version_tuple(text):
    nums = [int(x) for x in re.findall(r"\d+", text)[:3]]
    return tuple((nums + [0, 0, 0])[:3])

def installed_between(pkg, min_version=None, max_exclusive=None):
    try:
        current = version(pkg)
    except PackageNotFoundError:
        return False
    cur = version_tuple(current)
    if min_version is not None and cur < version_tuple(min_version):
        return False
    if max_exclusive is not None and cur >= version_tuple(max_exclusive):
        return False
    return True

def should_install_deps():
    mode = os.environ.get("INSTALL_DEPS", "auto").strip().lower()
    if mode in {"0", "false", "no", "skip"}:
        return False
    if mode in {"1", "true", "yes", "force"}:
        return True
    checks = [
        installed_between("transformers", "5.5.0"),
        installed_between("accelerate", "1.8.0"),
        installed_between("bitsandbytes", "0.49.0"),
        installed_between("huggingface_hub", "0.36.0"),
        installed_between("sentencepiece"),
        installed_between("protobuf", "5.29.1", "6.0.0"),
    ]
    return not all(checks)

if should_install_deps():
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-U", *PIP_PACKAGES])
else:
    print("Bỏ qua pip install vì dependency đã đủ hoặc INSTALL_DEPS=0.")

import torch
from huggingface_hub import login, notebook_login
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# ------------------------------------------------------------
# 1) TIỆN ÍCH
# ------------------------------------------------------------
def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True)
    except Exception as e:
        return f"CMD ERROR: {repr(e)}"

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def print_gpu(title="GPU"):
    print("=" * 110)
    print(title)
    print("=" * 110)
    print(run_cmd("nvidia-smi"))
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("Capability:", torch.cuda.get_device_capability(0))
        try:
            print("BF16 supported:", torch.cuda.is_bf16_supported())
        except Exception as e:
            print("BF16 supported check error:", repr(e))

def first_model_device(m):
    for p in m.parameters():
        return p.device
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def count_devices(m, name="model"):
    from collections import Counter
    c = Counter()
    total = 0
    try:
        for _, p in m.named_parameters():
            c[str(p.device)] += p.numel()
            total += p.numel()
        print(f"\n{name} device map:")
        for dev, n in c.items():
            print(f"  {dev}: {n/1e9:.3f}B params")
        print(f"  total visible: {total/1e9:.3f}B params")
        if any("cpu" in d.lower() for d in c):
            print("  CẢNH BÁO: Có parameter trên CPU -> sẽ rất chậm.")
        else:
            print("  OK: không thấy parameter trên CPU.")
    except Exception as e:
        print(f"Không đếm được device map cho {name}:", repr(e))

print_gpu("1) GPU TRƯỚC KHI LOAD")

# ------------------------------------------------------------
# 2) TỐI ƯU CƠ BẢN CHO T4
# ------------------------------------------------------------
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

cleanup_cuda()

# ------------------------------------------------------------
# 3) LOGIN HUGGING FACE
# ------------------------------------------------------------
print("=" * 110)
print("2) HUGGING FACE LOGIN")
print("=" * 110)

hf_token = os.environ.get("HF_TOKEN", None)

try:
    from google.colab import userdata
    try:
        hf_token = hf_token or userdata.get("HF_TOKEN")
    except Exception:
        pass
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    print("OK: đã login bằng HF_TOKEN.")
else:
    print("Không thấy HF_TOKEN trong env/Colab secrets.")
    print("Nếu tải model bị 403/rate limit, hãy login ở widget hiện ra.")
    try:
        notebook_login()
    except Exception as e:
        print("Bỏ qua notebook_login hoặc lỗi login:", repr(e))

# ------------------------------------------------------------
# 4) CHỌN MODEL
# ------------------------------------------------------------
# Mạnh nhất còn có cửa chạy T4:
TARGET_MODEL_ID = "google/gemma-4-E4B-it"
ASSISTANT_MODEL_ID = "google/gemma-4-E4B-it-assistant"

# Fallback nếu E4B + assistant không fit:
FALLBACK_TARGET_MODEL_ID = "google/gemma-4-E2B-it"
FALLBACK_ASSISTANT_MODEL_ID = "google/gemma-4-E2B-it-assistant"

# T4: FP16 thực dụng hơn BF16.
DTYPE = torch.float16
MTP_NUM_ASSISTANT_TOKENS = 8
MTP_ASSISTANT_SCHEDULE = "constant"
MTP_CONFIDENCE_THRESHOLD = 0.0
RUN_BASELINE_COMPARE = False
RUN_WARMUP = True
RUN_FULL_DFUZZ = True
RUN_SHORT_JSON = True

print("=" * 110)
print("3) CẤU HÌNH")
print("=" * 110)
print("Target ưu tiên:", TARGET_MODEL_ID)
print("Assistant ưu tiên:", ASSISTANT_MODEL_ID)
print("Fallback:", FALLBACK_TARGET_MODEL_ID)
print("dtype:", DTYPE)
print("MTP num_assistant_tokens:", MTP_NUM_ASSISTANT_TOKENS)
print("MTP schedule:", MTP_ASSISTANT_SCHEDULE)
print("MTP threshold:", MTP_CONFIDENCE_THRESHOLD)
print("RUN_BASELINE_COMPARE:", RUN_BASELINE_COMPARE)

# ------------------------------------------------------------
# 5) LOAD PROCESSOR
# ------------------------------------------------------------
def load_processor(model_id):
    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
    )
    tokenizer = getattr(processor, "tokenizer", processor)
    try:
        tokenizer.padding_side = "left"
    except Exception:
        pass
    try:
        if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
    except Exception:
        pass
    return processor

# ------------------------------------------------------------
# 6) LOAD MODEL TARGET / ASSISTANT
# ------------------------------------------------------------
def load_target_8bit(model_id):
    q = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=False,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_target_4bit(model_id):
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_assistant_4bit(model_id):
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_assistant_fp16(model_id):
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def configure_mtp_assistant(m):
    if m is None:
        return False
    cfg = getattr(m, "generation_config", None)
    if cfg is None:
        return False
    cfg.num_assistant_tokens = MTP_NUM_ASSISTANT_TOKENS
    cfg.num_assistant_tokens_schedule = MTP_ASSISTANT_SCHEDULE
    cfg.assistant_confidence_threshold = MTP_CONFIDENCE_THRESHOLD
    return True

def load_stack(target_id, assistant_id):
    """
    Chiến lược:
    1. Target E4B 8-bit vì chất lượng tốt hơn 4-bit.
    2. Assistant 4-bit để cố fit VRAM T4.
    3. Nếu OOM: target E4B 4-bit + assistant 4-bit.
    4. Nếu vẫn fail: fallback E2B 8-bit + assistant 4-bit.
    5. Nếu assistant fail: vẫn giữ target để benchmark baseline.
    """
    global processor, model, assistant_model, active_target_id, active_assistant_id, target_mode, assistant_mode

    processor = load_processor(target_id)
    active_target_id = target_id
    active_assistant_id = assistant_id
    model = None
    assistant_model = None
    target_mode = None
    assistant_mode = None

    print("=" * 110)
    print("4) LOAD TARGET MODEL")
    print("=" * 110)

    try:
        print("Thử target 8-bit:", target_id)
        model = load_target_8bit(target_id)
        target_mode = "8bit"
        print("OK: target 8-bit")
    except Exception as e1:
        print("FAIL target 8-bit:", repr(e1))
        traceback.print_exc(limit=1)
        cleanup_cuda()
        print("Thử target 4-bit:", target_id)
        model = load_target_4bit(target_id)
        target_mode = "4bit_nf4"
        print("OK: target 4-bit")

    count_devices(model, "target")
    print_gpu("GPU SAU KHI LOAD TARGET")

    print("=" * 110)
    print("5) LOAD ASSISTANT MODEL CHO MTP")
    print("=" * 110)

    # Trên T4 ưu tiên assistant 4-bit.
    try:
        print("Thử assistant 4-bit:", assistant_id)
        assistant_model = load_assistant_4bit(assistant_id)
        assistant_mode = "4bit_nf4"
        print("OK: assistant 4-bit")
        if configure_mtp_assistant(assistant_model):
            print(
                "OK: MTP strict",
                f"n={MTP_NUM_ASSISTANT_TOKENS}",
                f"schedule={MTP_ASSISTANT_SCHEDULE}",
                f"threshold={MTP_CONFIDENCE_THRESHOLD}",
            )
    except Exception as e:
        print("FAIL assistant 4-bit:", repr(e))
        traceback.print_exc(limit=1)
        cleanup_cuda()
        assistant_model = None
        assistant_mode = None
        print("Không có assistant -> vẫn benchmark baseline target.")

    if assistant_model is not None:
        count_devices(assistant_model, "assistant")
    print_gpu("GPU SAU KHI LOAD ASSISTANT")

try:
    load_stack(TARGET_MODEL_ID, ASSISTANT_MODEL_ID)
except Exception as e:
    print("\n" + "!" * 110)
    print("E4B STACK FAIL. FALLBACK SANG E2B STACK.")
    print("Lỗi:", repr(e))
    print("!" * 110)
    traceback.print_exc(limit=2)
    cleanup_cuda()
    load_stack(FALLBACK_TARGET_MODEL_ID, FALLBACK_ASSISTANT_MODEL_ID)

# ------------------------------------------------------------
# 7) PROMPT DFUZZ GỌN
# ------------------------------------------------------------
SYSTEM_PROMPT = (
    "Bạn là bộ phân tích mã C++ cho fuzzing API học sâu. "
    "Chỉ trả JSON hợp lệ. Không markdown. Không giải thích."
)

SCHEMA_MIN = {
    "api": "string",
    "params": [{"n": "string", "t": "Tensor|Int|Bool|Str|Float|Scalar|List|Optional|Unknown"}],
    "cases": [{"cond": "string", "bad": "string", "p": ["string"], "t": ["string"], "conf": 0.0}]
}

DFUZZ_PROMPT = f"""
Phân tích check C++ và rút ca biên vi phạm.

Luật:
- Chỉ dùng type: Tensor, Int, Bool, Str, Float, Scalar, List, Optional, Unknown.
- JSON phải đúng schema.
- Viết cực ngắn.

Schema:
{json.dumps(SCHEMA_MIN, ensure_ascii=False)}

Signature:
TORCH_META_FUNC(upsample_bicubic2d)(
    const Tensor& input,
    IntArrayRef output_size,
    bool align_corners,
    c10::optional<double> scales_h,
    c10::optional<double> scales_w)

Check:
TORCH_CHECK(
    input.numel() != 0 ||
    c10::multiply_integers(input.sizes().begin() + 1, input.sizes().end()),
    "Non-empty 4D data tensor expected but got a tensor with sizes ",
    input.sizes()
);
""".strip()

SHORT_PROMPT = 'Trả JSON hợp lệ: {"ok":true,"edge":"Tensor rỗng"}'

# ------------------------------------------------------------
# 8) TOKENIZE / GENERATE / BENCHMARK
# ------------------------------------------------------------
def make_inputs(user_prompt, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    dev = first_model_device(model)
    inputs = {
        k: v.to(dev) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }
    return inputs

def extract_first_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    if start < 0:
        raise ValueError("Không thấy JSON object")

    depth = 0
    in_str = False
    esc = False

    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]

    raise ValueError("JSON chưa đóng ngoặc")

def parse_json_loose(text):
    return json.loads(extract_first_json_object(text))

def generate_once(prompt, max_new_tokens=160, use_mtp=False, use_static_cache=True):
    inputs = make_inputs(prompt)
    input_tokens = int(inputs["input_ids"].shape[-1])

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    if use_static_cache:
        gen_kwargs["cache_implementation"] = "static"

    if use_mtp and assistant_model is not None:
        configure_mtp_assistant(assistant_model)
        gen_kwargs["assistant_model"] = assistant_model

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    used_static = use_static_cache
    used_mtp = bool(use_mtp and assistant_model is not None)

    try:
        with torch.inference_mode():
            out = model.generate(**gen_kwargs)
    except Exception as e:
        msg = str(e).lower()
        # Một số backend/model chưa nhận static cache hoặc assistant + static cùng lúc.
        if "cache_implementation" in msg or "static" in msg:
            print("Static cache lỗi -> fallback dynamic cache:", repr(e))
            gen_kwargs.pop("cache_implementation", None)
            used_static = False
            with torch.inference_mode():
                out = model.generate(**gen_kwargs)
        elif used_mtp:
            print("MTP lỗi -> fallback baseline không assistant:", repr(e))
            gen_kwargs.pop("assistant_model", None)
            used_mtp = False
            with torch.inference_mode():
                out = model.generate(**gen_kwargs)
        else:
            raise

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t1 = time.perf_counter()

    output_tokens = int(out[0].shape[-1] - input_tokens)
    elapsed = t1 - t0
    tok_s = output_tokens / elapsed if elapsed > 0 else 0.0
    decoded = processor.decode(out[0][input_tokens:], skip_special_tokens=True)

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "seconds": elapsed,
        "tok_s": tok_s,
        "used_static_cache": used_static,
        "used_mtp": used_mtp,
        "text": decoded,
    }

def bench(name, prompt, max_new_tokens=160, use_mtp=False, parse=False):
    print("\n" + "-" * 110)
    print("TEST:", name)
    print("-" * 110)

    inp = make_inputs(prompt)
    print("active_target:", active_target_id)
    print("target_mode:", target_mode)
    print("active_assistant:", active_assistant_id if assistant_model is not None else None)
    print("assistant_mode:", assistant_mode)
    print("prompt_chars:", len(prompt))
    print("prompt_tokens:", int(inp["input_ids"].shape[-1]))
    print("max_new_tokens:", max_new_tokens)
    print("use_mtp_requested:", use_mtp)
    if use_mtp and assistant_model is not None:
        cfg = getattr(assistant_model, "generation_config", None)
        if cfg is not None:
            print("mtp_num_assistant_tokens:", getattr(cfg, "num_assistant_tokens", None))
            print("mtp_schedule:", getattr(cfg, "num_assistant_tokens_schedule", None))
            print("mtp_threshold:", getattr(cfg, "assistant_confidence_threshold", None))

    r = generate_once(
        prompt,
        max_new_tokens=max_new_tokens,
        use_mtp=use_mtp,
        use_static_cache=True,
    )

    print("input_tokens:", r["input_tokens"])
    print("output_tokens:", r["output_tokens"])
    print("seconds:", round(r["seconds"], 3))
    print("tokens_per_second:", round(r["tok_s"], 3))
    print("used_static_cache:", r["used_static_cache"])
    print("used_mtp:", r["used_mtp"])
    print("preview:")
    print(r["text"][:1600])

    if parse:
        try:
            obj = parse_json_loose(r["text"])
            print("\nJSON PARSE: OK")
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:2500])
        except Exception as e:
            print("\nJSON PARSE: LỖI:", repr(e))

    return r

# ------------------------------------------------------------
# 9) CHẠY BENCHMARK
# ------------------------------------------------------------
print("=" * 110)
print("6) BENCHMARK FAST MTP / BASELINE OPTIONAL")
print("=" * 110)

results = []

fast_use_mtp = assistant_model is not None

if RUN_WARMUP:
    warmup_name = "warmup_mtp_short" if fast_use_mtp else "warmup_baseline_short"
    results.append((warmup_name, bench(
        warmup_name,
        SHORT_PROMPT,
        max_new_tokens=48,
        use_mtp=fast_use_mtp,
        parse=False,
    )))

if RUN_BASELINE_COMPARE:
    results.append(("baseline_dfuzz", bench(
        "baseline_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=False,
        parse=True,
    )))
else:
    print("\nFAST MODE: bỏ qua baseline_dfuzz. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.")

if RUN_FULL_DFUZZ and assistant_model is not None:
    results.append(("mtp_dfuzz", bench(
        "mtp_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=True,
        parse=True,
    )))
elif RUN_FULL_DFUZZ and not RUN_BASELINE_COMPARE:
    results.append(("baseline_dfuzz", bench(
        "baseline_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=False,
        parse=True,
    )))
elif RUN_FULL_DFUZZ:
    print("\nKhông có assistant_model -> bỏ qua MTP benchmark.")

# Một lượt output ngắn hơn, vì DFUZZ thực tế nên ép JSON ngắn.
SHORT_JSON_DFUZZ_PROMPT = """
Rút ca biên từ check C++ sau. Chỉ trả JSON một dòng.
Schema: {"api":"str","bad":"str","p":["str"],"t":["str"],"conf":0.0}
Signature: upsample_bicubic2d(Tensor input, List output_size, Bool align_corners, Optional scales_h, Optional scales_w)
Check: input.numel()!=0 || product(input.sizes()[1:]) != 0
""".strip()

if RUN_BASELINE_COMPARE:
    results.append(("baseline_short_json", bench(
        "baseline_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=False,
        parse=True,
    )))
else:
    print("\nFAST MODE: bỏ qua baseline_short_json. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.")

if RUN_SHORT_JSON and assistant_model is not None:
    results.append(("mtp_short_json", bench(
        "mtp_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=True,
        parse=True,
    )))
elif RUN_SHORT_JSON and not RUN_BASELINE_COMPARE:
    results.append(("baseline_short_json", bench(
        "baseline_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=False,
        parse=True,
    )))

# ------------------------------------------------------------
# 10) TỔNG KẾT
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("7) TỔNG KẾT")
print("=" * 110)

summary = []
for name, r in results:
    summary.append({
        "name": name,
        "in_tok": r["input_tokens"],
        "out_tok": r["output_tokens"],
        "seconds": round(r["seconds"], 3),
        "tok_s": round(r["tok_s"], 3),
        "static": r["used_static_cache"],
        "mtp": r["used_mtp"],
    })

print(json.dumps(summary, ensure_ascii=False, indent=2))

base = next((r for n, r in results if n == "baseline_dfuzz"), None)
mtp = next((r for n, r in results if n == "mtp_dfuzz"), None)

if base and mtp:
    speedup = mtp["tok_s"] / base["tok_s"] if base["tok_s"] > 0 else 0
    time_ratio = base["seconds"] / mtp["seconds"] if mtp["seconds"] > 0 else 0
    print("\nSo sánh DFUZZ:")
    print("baseline tok/s:", round(base["tok_s"], 3))
    print("MTP tok/s:", round(mtp["tok_s"], 3))
    print("tok/s speedup:", round(speedup, 3), "x")
    print("wall-time speedup:", round(time_ratio, 3), "x")

meaningful_summary = [x for x in summary if not x["name"].startswith("warmup")]
if meaningful_summary:
    fastest_tok = max(meaningful_summary, key=lambda x: x["tok_s"])
    fastest_wall = min(meaningful_summary, key=lambda x: x["seconds"])
    print("\nNhanh nhất:")
    print("tok/s:", fastest_tok["name"], fastest_tok["tok_s"], "tok/s")
    print("wall-time:", fastest_wall["name"], fastest_wall["seconds"], "s")

print("\nKết luận đọc nhanh:")
print("- Mặc định fast mode: target hiện tại + assistant_model + MTP strict n=8/constant/0.0.")
print("- Muốn so baseline lại: đặt RUN_BASELINE_COMPARE=True.")
print("- Nếu MTP OOM/lỗi: script tự rơi về baseline cho test cần thiết.")
print("- Nếu active_target là E4B: đây là hướng mạnh nhất còn có cửa trên T4.")
print("- Nếu fallback sang E2B: E4B không fit môi trường, dùng E2B lọc hàng loạt rồi E4B/A100 kiểm lại case khó.")
print("- Nếu tok/s vẫn thấp dù VRAM thừa: cổ chai là T4 + decoding/quant kernel, không phải thiếu VRAM.")
print_gpu("GPU CUỐI CELL")

0) CÀI / CẬP NHẬT THƯ VIỆN
1) GPU TRƯỚC KHI LOAD
Thu May 21 03:09:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


3) CẤU HÌNH
Target ưu tiên: google/gemma-4-E4B-it
Assistant ưu tiên: google/gemma-4-E4B-it-assistant
Fallback: google/gemma-4-E2B-it
dtype: torch.float16
MTP num_assistant_tokens: 8
MTP schedule: constant
MTP threshold: 0.0
RUN_BASELINE_COMPARE: False


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

OK: assistant 4-bit
OK: MTP strict n=8 schedule=constant threshold=0.0

assistant device map:
  cuda:0: 0.073B params
  total visible: 0.073B params
  OK: không thấy parameter trên CPU.
GPU SAU KHI LOAD ASSISTANT
Thu May 21 03:15:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0  

[transformers] An assistant model is provided, using a dynamic cache instead of a cache of type='static'.


input_tokens: 59
output_tokens: 12
seconds: 2.76
tokens_per_second: 4.348
used_static_cache: True
used_mtp: True
preview:
{"ok":true,"edge":"Tensor rỗng"}

FAST MODE: bỏ qua baseline_dfuzz. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.

--------------------------------------------------------------------------------------------------------------
TEST: mtp_dfuzz
--------------------------------------------------------------------------------------------------------------
active_target: google/gemma-4-E4B-it
target_mode: 8bit
active_assistant: google/gemma-4-E4B-it-assistant
assistant_mode: 4bit_nf4
prompt_chars: 800
prompt_tokens: 319
max_new_tokens: 180
use_mtp_requested: True
mtp_num_assistant_tokens: 8
mtp_schedule: constant
mtp_threshold: 0.0
input_tokens: 319
output_tokens: 173
seconds: 17.095
tokens_per_second: 10.12
used_static_cache: True
used_mtp: True
preview:
{"api": "upsample_bicubic2d", "params": [{"n": "input", "t": "Tensor"}, {"n": "output_size", "t": "List"}, {"n": "alig

In [ ]:
# ============================================================
# GEMMA 4 E4B-it + MTP ASSISTANT OPTIMIZED TEST FOR COLAB T4
# Một cell duy nhất:
# cài -> login HF -> load E4B target -> load assistant -> benchmark baseline vs MTP -> test DFUZZ JSON
# ============================================================

import os, sys, gc, re, json, time, subprocess, traceback
from importlib.metadata import PackageNotFoundError, version

print("=" * 110)
print("0) CÀI / CẬP NHẬT THƯ VIỆN")
print("=" * 110)

PIP_PACKAGES = [
    "transformers>=5.5.0",
    "accelerate>=1.8.0",
    "bitsandbytes>=0.49.0",
    "huggingface_hub>=0.36.0",
    "sentencepiece",
    "protobuf<6,>=5.29.1",
]

def version_tuple(text):
    nums = [int(x) for x in re.findall(r"\d+", text)[:3]]
    return tuple((nums + [0, 0, 0])[:3])

def installed_between(pkg, min_version=None, max_exclusive=None):
    try:
        current = version(pkg)
    except PackageNotFoundError:
        return False
    cur = version_tuple(current)
    if min_version is not None and cur < version_tuple(min_version):
        return False
    if max_exclusive is not None and cur >= version_tuple(max_exclusive):
        return False
    return True

def should_install_deps():
    mode = os.environ.get("INSTALL_DEPS", "auto").strip().lower()
    if mode in {"0", "false", "no", "skip"}:
        return False
    if mode in {"1", "true", "yes", "force"}:
        return True
    checks = [
        installed_between("transformers", "5.5.0"),
        installed_between("accelerate", "1.8.0"),
        installed_between("bitsandbytes", "0.49.0"),
        installed_between("huggingface_hub", "0.36.0"),
        installed_between("sentencepiece"),
        installed_between("protobuf", "5.29.1", "6.0.0"),
    ]
    return not all(checks)

def env_bool(name, default):
    raw = os.environ.get(name, "1" if default else "0").strip().lower()
    return raw not in {"0", "false", "no", "skip"}

def env_float(name, default):
    raw = os.environ.get(name, str(default)).strip()
    try:
        return float(raw)
    except Exception:
        print(f"{name} không hợp lệ: {raw!r}; dùng default {default}.")
        return float(default)

def env_int_list(name, default):
    raw = os.environ.get(name, default).strip()
    vals = []
    for part in raw.split(","):
        part = part.strip()
        if not part:
            continue
        try:
            vals.append(int(part))
        except Exception:
            print(f"Bỏ qua {name} item không hợp lệ: {part!r}")
    return vals or [int(x.strip()) for x in default.split(",") if x.strip()]

if should_install_deps():
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-U", *PIP_PACKAGES])
else:
    print("Bỏ qua pip install vì dependency đã đủ hoặc INSTALL_DEPS=0.")

import torch
from huggingface_hub import login, notebook_login
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# ------------------------------------------------------------
# 1) TIỆN ÍCH
# ------------------------------------------------------------
def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True)
    except Exception as e:
        return f"CMD ERROR: {repr(e)}"

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def print_gpu(title="GPU"):
    print("=" * 110)
    print(title)
    print("=" * 110)
    print(run_cmd("nvidia-smi"))
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("Capability:", torch.cuda.get_device_capability(0))
        try:
            print("BF16 supported:", torch.cuda.is_bf16_supported())
        except Exception as e:
            print("BF16 supported check error:", repr(e))

def first_model_device(m):
    for p in m.parameters():
        return p.device
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def count_devices(m, name="model"):
    from collections import Counter
    c = Counter()
    total = 0
    try:
        for _, p in m.named_parameters():
            c[str(p.device)] += p.numel()
            total += p.numel()
        print(f"\n{name} device map:")
        for dev, n in c.items():
            print(f"  {dev}: {n/1e9:.3f}B params")
        print(f"  total visible: {total/1e9:.3f}B params")
        if any("cpu" in d.lower() for d in c):
            print("  CẢNH BÁO: Có parameter trên CPU -> sẽ rất chậm.")
        else:
            print("  OK: không thấy parameter trên CPU.")
    except Exception as e:
        print(f"Không đếm được device map cho {name}:", repr(e))

print_gpu("1) GPU TRƯỚC KHI LOAD")

# ------------------------------------------------------------
# 2) TỐI ƯU CƠ BẢN CHO T4
# ------------------------------------------------------------
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

cleanup_cuda()

# ------------------------------------------------------------
# 3) LOGIN HUGGING FACE
# ------------------------------------------------------------
print("=" * 110)
print("2) HUGGING FACE LOGIN")
print("=" * 110)

hf_token = os.environ.get("HF_TOKEN", None)

try:
    from google.colab import userdata
    try:
        hf_token = hf_token or userdata.get("HF_TOKEN")
    except Exception:
        pass
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    print("OK: đã login bằng HF_TOKEN.")
else:
    print("Không thấy HF_TOKEN trong env/Colab secrets.")
    print("Nếu tải model bị 403/rate limit, hãy login ở widget hiện ra.")
    try:
        notebook_login()
    except Exception as e:
        print("Bỏ qua notebook_login hoặc lỗi login:", repr(e))

# ------------------------------------------------------------
# 4) CHỌN MODEL
# ------------------------------------------------------------
# Mạnh nhất còn có cửa chạy T4:
TARGET_MODEL_ID = "google/gemma-4-E4B-it"
ASSISTANT_MODEL_ID = "google/gemma-4-E4B-it-assistant"

# Fallback nếu E4B + assistant không fit:
FALLBACK_TARGET_MODEL_ID = "google/gemma-4-E2B-it"
FALLBACK_ASSISTANT_MODEL_ID = "google/gemma-4-E2B-it-assistant"

# T4: FP16 thực dụng hơn BF16.
DTYPE = torch.float16
TARGET_INT8_THRESHOLD = env_float("TARGET_INT8_THRESHOLD", 0.0)
MTP_NUM_ASSISTANT_TOKENS = 8
MTP_ASSISTANT_SCHEDULE = "constant"
MTP_CONFIDENCE_THRESHOLD = 0.0
RUN_MTP_SWEEP = env_bool("RUN_MTP_SWEEP", True)
MTP_SWEEP_TOKENS = env_int_list("MTP_SWEEP_TOKENS", "4,8,12,16,24")
RUN_BASELINE_COMPARE = False
RUN_WARMUP = True
RUN_FULL_DFUZZ = True
RUN_SHORT_JSON = True

print("=" * 110)
print("3) CẤU HÌNH")
print("=" * 110)
print("Target ưu tiên:", TARGET_MODEL_ID)
print("Assistant ưu tiên:", ASSISTANT_MODEL_ID)
print("Fallback:", FALLBACK_TARGET_MODEL_ID)
print("dtype:", DTYPE)
print("TARGET_INT8_THRESHOLD:", TARGET_INT8_THRESHOLD)
print("MTP num_assistant_tokens:", MTP_NUM_ASSISTANT_TOKENS)
print("MTP schedule:", MTP_ASSISTANT_SCHEDULE)
print("MTP threshold:", MTP_CONFIDENCE_THRESHOLD)
print("RUN_MTP_SWEEP:", RUN_MTP_SWEEP)
print("MTP_SWEEP_TOKENS:", MTP_SWEEP_TOKENS)
print("RUN_BASELINE_COMPARE:", RUN_BASELINE_COMPARE)

# ------------------------------------------------------------
# 5) LOAD PROCESSOR
# ------------------------------------------------------------
def load_processor(model_id):
    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
    )
    tokenizer = getattr(processor, "tokenizer", processor)
    try:
        tokenizer.padding_side = "left"
    except Exception:
        pass
    try:
        if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
    except Exception:
        pass
    return processor

# ------------------------------------------------------------
# 6) LOAD MODEL TARGET / ASSISTANT
# ------------------------------------------------------------
def load_target_8bit(model_id):
    q = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=TARGET_INT8_THRESHOLD,
        llm_int8_enable_fp32_cpu_offload=False,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_target_4bit(model_id):
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_assistant_4bit(model_id):
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_assistant_fp16(model_id):
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def configure_mtp_assistant(m, num_tokens=None, schedule=None, threshold=None):
    if m is None:
        return False
    cfg = getattr(m, "generation_config", None)
    if cfg is None:
        return False
    cfg.num_assistant_tokens = MTP_NUM_ASSISTANT_TOKENS if num_tokens is None else int(num_tokens)
    cfg.num_assistant_tokens_schedule = MTP_ASSISTANT_SCHEDULE if schedule is None else schedule
    cfg.assistant_confidence_threshold = MTP_CONFIDENCE_THRESHOLD if threshold is None else float(threshold)
    return True

def load_stack(target_id, assistant_id):
    """
    Chiến lược:
    1. Target E4B 8-bit vì chất lượng tốt hơn 4-bit.
    2. Assistant 4-bit để cố fit VRAM T4.
    3. Nếu OOM: target E4B 4-bit + assistant 4-bit.
    4. Nếu vẫn fail: fallback E2B 8-bit + assistant 4-bit.
    5. Nếu assistant fail: vẫn giữ target để benchmark baseline.
    """
    global processor, model, assistant_model, active_target_id, active_assistant_id, target_mode, assistant_mode

    processor = load_processor(target_id)
    active_target_id = target_id
    active_assistant_id = assistant_id
    model = None
    assistant_model = None
    target_mode = None
    assistant_mode = None

    print("=" * 110)
    print("4) LOAD TARGET MODEL")
    print("=" * 110)

    try:
        print("Thử target 8-bit:", target_id)
        model = load_target_8bit(target_id)
        target_mode = "8bit"
        print("OK: target 8-bit")
    except Exception as e1:
        print("FAIL target 8-bit:", repr(e1))
        traceback.print_exc(limit=1)
        cleanup_cuda()
        print("Thử target 4-bit:", target_id)
        model = load_target_4bit(target_id)
        target_mode = "4bit_nf4"
        print("OK: target 4-bit")

    count_devices(model, "target")
    print_gpu("GPU SAU KHI LOAD TARGET")

    print("=" * 110)
    print("5) LOAD ASSISTANT MODEL CHO MTP")
    print("=" * 110)

    # Trên T4 ưu tiên assistant 4-bit.
    try:
        print("Thử assistant 4-bit:", assistant_id)
        assistant_model = load_assistant_4bit(assistant_id)
        assistant_mode = "4bit_nf4"
        print("OK: assistant 4-bit")
        if configure_mtp_assistant(assistant_model):
            print(
                "OK: MTP strict",
                f"n={MTP_NUM_ASSISTANT_TOKENS}",
                f"schedule={MTP_ASSISTANT_SCHEDULE}",
                f"threshold={MTP_CONFIDENCE_THRESHOLD}",
            )
    except Exception as e:
        print("FAIL assistant 4-bit:", repr(e))
        traceback.print_exc(limit=1)
        cleanup_cuda()
        assistant_model = None
        assistant_mode = None
        print("Không có assistant -> vẫn benchmark baseline target.")

    if assistant_model is not None:
        count_devices(assistant_model, "assistant")
    print_gpu("GPU SAU KHI LOAD ASSISTANT")

try:
    load_stack(TARGET_MODEL_ID, ASSISTANT_MODEL_ID)
except Exception as e:
    print("\n" + "!" * 110)
    print("E4B STACK FAIL. FALLBACK SANG E2B STACK.")
    print("Lỗi:", repr(e))
    print("!" * 110)
    traceback.print_exc(limit=2)
    cleanup_cuda()
    load_stack(FALLBACK_TARGET_MODEL_ID, FALLBACK_ASSISTANT_MODEL_ID)

# ------------------------------------------------------------
# 7) PROMPT DFUZZ GỌN
# ------------------------------------------------------------
SYSTEM_PROMPT = (
    "Bạn là bộ phân tích mã C++ cho fuzzing API học sâu. "
    "Chỉ trả JSON hợp lệ. Không markdown. Không giải thích."
)

SCHEMA_MIN = {
    "api": "string",
    "params": [{"n": "string", "t": "Tensor|Int|Bool|Str|Float|Scalar|List|Optional|Unknown"}],
    "cases": [{"cond": "string", "bad": "string", "p": ["string"], "t": ["string"], "conf": 0.0}]
}

DFUZZ_PROMPT = f"""
Phân tích check C++ và rút ca biên vi phạm.

Luật:
- Chỉ dùng type: Tensor, Int, Bool, Str, Float, Scalar, List, Optional, Unknown.
- JSON phải đúng schema.
- Viết cực ngắn.

Schema:
{json.dumps(SCHEMA_MIN, ensure_ascii=False)}

Signature:
TORCH_META_FUNC(upsample_bicubic2d)(
    const Tensor& input,
    IntArrayRef output_size,
    bool align_corners,
    c10::optional<double> scales_h,
    c10::optional<double> scales_w)

Check:
TORCH_CHECK(
    input.numel() != 0 ||
    c10::multiply_integers(input.sizes().begin() + 1, input.sizes().end()),
    "Non-empty 4D data tensor expected but got a tensor with sizes ",
    input.sizes()
);
""".strip()

SHORT_PROMPT = 'Trả JSON hợp lệ: {"ok":true,"edge":"Tensor rỗng"}'

# ------------------------------------------------------------
# 8) TOKENIZE / GENERATE / BENCHMARK
# ------------------------------------------------------------
def make_inputs(user_prompt, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    dev = first_model_device(model)
    inputs = {
        k: v.to(dev) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }
    return inputs

def extract_first_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    if start < 0:
        raise ValueError("Không thấy JSON object")

    depth = 0
    in_str = False
    esc = False

    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]

    raise ValueError("JSON chưa đóng ngoặc")

def parse_json_loose(text):
    return json.loads(extract_first_json_object(text))

def generate_once(
    prompt,
    max_new_tokens=160,
    use_mtp=False,
    use_static_cache=True,
    mtp_tokens=None,
    mtp_schedule=None,
    mtp_threshold=None,
):
    inputs = make_inputs(prompt)
    input_tokens = int(inputs["input_ids"].shape[-1])

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    if use_static_cache:
        gen_kwargs["cache_implementation"] = "static"

    if use_mtp and assistant_model is not None:
        configure_mtp_assistant(assistant_model, mtp_tokens, mtp_schedule, mtp_threshold)
        gen_kwargs["assistant_model"] = assistant_model

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    used_mtp = bool(use_mtp and assistant_model is not None)
    # Transformers currently switches assisted decoding to dynamic cache even if static was requested.
    used_static = bool(use_static_cache and not used_mtp)

    try:
        with torch.inference_mode():
            out = model.generate(**gen_kwargs)
    except Exception as e:
        msg = str(e).lower()
        # Một số backend/model chưa nhận static cache hoặc assistant + static cùng lúc.
        if "cache_implementation" in msg or "static" in msg:
            print("Static cache lỗi -> fallback dynamic cache:", repr(e))
            gen_kwargs.pop("cache_implementation", None)
            used_static = False
            with torch.inference_mode():
                out = model.generate(**gen_kwargs)
        elif used_mtp:
            print("MTP lỗi -> fallback baseline không assistant:", repr(e))
            gen_kwargs.pop("assistant_model", None)
            used_mtp = False
            with torch.inference_mode():
                out = model.generate(**gen_kwargs)
        else:
            raise

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t1 = time.perf_counter()

    output_tokens = int(out[0].shape[-1] - input_tokens)
    elapsed = t1 - t0
    tok_s = output_tokens / elapsed if elapsed > 0 else 0.0
    decoded = processor.decode(out[0][input_tokens:], skip_special_tokens=True)

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "seconds": elapsed,
        "tok_s": tok_s,
        "used_static_cache": used_static,
        "used_mtp": used_mtp,
        "text": decoded,
    }

def bench(
    name,
    prompt,
    max_new_tokens=160,
    use_mtp=False,
    parse=False,
    mtp_tokens=None,
    mtp_schedule=None,
    mtp_threshold=None,
):
    print("\n" + "-" * 110)
    print("TEST:", name)
    print("-" * 110)

    inp = make_inputs(prompt)
    print("active_target:", active_target_id)
    print("target_mode:", target_mode)
    print("active_assistant:", active_assistant_id if assistant_model is not None else None)
    print("assistant_mode:", assistant_mode)
    print("prompt_chars:", len(prompt))
    print("prompt_tokens:", int(inp["input_ids"].shape[-1]))
    print("max_new_tokens:", max_new_tokens)
    print("use_mtp_requested:", use_mtp)
    if use_mtp and assistant_model is not None:
        configure_mtp_assistant(assistant_model, mtp_tokens, mtp_schedule, mtp_threshold)
        cfg = getattr(assistant_model, "generation_config", None)
        if cfg is not None:
            print("mtp_num_assistant_tokens:", getattr(cfg, "num_assistant_tokens", None))
            print("mtp_schedule:", getattr(cfg, "num_assistant_tokens_schedule", None))
            print("mtp_threshold:", getattr(cfg, "assistant_confidence_threshold", None))

    r = generate_once(
        prompt,
        max_new_tokens=max_new_tokens,
        use_mtp=use_mtp,
        use_static_cache=True,
        mtp_tokens=mtp_tokens,
        mtp_schedule=mtp_schedule,
        mtp_threshold=mtp_threshold,
    )

    print("input_tokens:", r["input_tokens"])
    print("output_tokens:", r["output_tokens"])
    print("seconds:", round(r["seconds"], 3))
    print("tokens_per_second:", round(r["tok_s"], 3))
    print("used_static_cache:", r["used_static_cache"])
    print("used_mtp:", r["used_mtp"])
    print("preview:")
    print(r["text"][:1600])

    if parse:
        try:
            obj = parse_json_loose(r["text"])
            print("\nJSON PARSE: OK")
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:2500])
        except Exception as e:
            print("\nJSON PARSE: LỖI:", repr(e))

    return r

# ------------------------------------------------------------
# 9) CHẠY BENCHMARK
# ------------------------------------------------------------
print("=" * 110)
print("6) BENCHMARK FAST MTP / BASELINE OPTIONAL")
print("=" * 110)

results = []

fast_use_mtp = assistant_model is not None

if RUN_WARMUP:
    warmup_name = "warmup_mtp_short" if fast_use_mtp else "warmup_baseline_short"
    results.append((warmup_name, bench(
        warmup_name,
        SHORT_PROMPT,
        max_new_tokens=48,
        use_mtp=fast_use_mtp,
        parse=False,
    )))

if RUN_BASELINE_COMPARE:
    results.append(("baseline_dfuzz", bench(
        "baseline_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=False,
        parse=True,
    )))
else:
    print("\nFAST MODE: bỏ qua baseline_dfuzz. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.")

if RUN_FULL_DFUZZ and assistant_model is not None:
    results.append(("mtp_dfuzz", bench(
        "mtp_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=True,
        parse=True,
    )))
elif RUN_FULL_DFUZZ and not RUN_BASELINE_COMPARE:
    results.append(("baseline_dfuzz", bench(
        "baseline_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=False,
        parse=True,
    )))
elif RUN_FULL_DFUZZ:
    print("\nKhông có assistant_model -> bỏ qua MTP benchmark.")

# Một lượt output ngắn hơn, vì DFUZZ thực tế nên ép JSON ngắn.
SHORT_JSON_DFUZZ_PROMPT = """
Rút ca biên từ check C++ sau. Chỉ trả JSON một dòng.
Schema: {"api":"str","bad":"str","p":["str"],"t":["str"],"conf":0.0}
Signature: upsample_bicubic2d(Tensor input, List output_size, Bool align_corners, Optional scales_h, Optional scales_w)
Check: input.numel()!=0 || product(input.sizes()[1:]) != 0
""".strip()

if RUN_BASELINE_COMPARE:
    results.append(("baseline_short_json", bench(
        "baseline_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=False,
        parse=True,
    )))
else:
    print("\nFAST MODE: bỏ qua baseline_short_json. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.")

if RUN_SHORT_JSON and assistant_model is not None and RUN_MTP_SWEEP:
    seen_mtp_tokens = set()
    for n in MTP_SWEEP_TOKENS:
        if n in seen_mtp_tokens:
            continue
        seen_mtp_tokens.add(n)
        name = f"mtp_short_json_n{n}"
        results.append((name, bench(
            name,
            SHORT_JSON_DFUZZ_PROMPT,
            max_new_tokens=96,
            use_mtp=True,
            parse=True,
            mtp_tokens=n,
        )))
elif RUN_SHORT_JSON and assistant_model is not None:
    results.append(("mtp_short_json", bench(
        "mtp_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=True,
        parse=True,
    )))
elif RUN_SHORT_JSON and not RUN_BASELINE_COMPARE:
    results.append(("baseline_short_json", bench(
        "baseline_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=False,
        parse=True,
    )))

# ------------------------------------------------------------
# 10) TỔNG KẾT
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("7) TỔNG KẾT")
print("=" * 110)

summary = []
for name, r in results:
    summary.append({
        "name": name,
        "in_tok": r["input_tokens"],
        "out_tok": r["output_tokens"],
        "seconds": round(r["seconds"], 3),
        "tok_s": round(r["tok_s"], 3),
        "static": r["used_static_cache"],
        "mtp": r["used_mtp"],
    })

print(json.dumps(summary, ensure_ascii=False, indent=2))

base = next((r for n, r in results if n == "baseline_dfuzz"), None)
mtp = next((r for n, r in results if n == "mtp_dfuzz"), None)

if base and mtp:
    speedup = mtp["tok_s"] / base["tok_s"] if base["tok_s"] > 0 else 0
    time_ratio = base["seconds"] / mtp["seconds"] if mtp["seconds"] > 0 else 0
    print("\nSo sánh DFUZZ:")
    print("baseline tok/s:", round(base["tok_s"], 3))
    print("MTP tok/s:", round(mtp["tok_s"], 3))
    print("tok/s speedup:", round(speedup, 3), "x")
    print("wall-time speedup:", round(time_ratio, 3), "x")

meaningful_summary = [x for x in summary if not x["name"].startswith("warmup")]
if meaningful_summary:
    fastest_tok = max(meaningful_summary, key=lambda x: x["tok_s"])
    fastest_wall = min(meaningful_summary, key=lambda x: x["seconds"])
    print("\nNhanh nhất:")
    print("tok/s:", fastest_tok["name"], fastest_tok["tok_s"], "tok/s")
    print("wall-time:", fastest_wall["name"], fastest_wall["seconds"], "s")

print("\nKết luận đọc nhanh:")
print("- Mặc định fast mode: target hiện tại + assistant_model + MTP strict n=8/constant/0.0.")
print("- Muốn so baseline lại: đặt RUN_BASELINE_COMPARE=True.")
print("- Nếu MTP OOM/lỗi: script tự rơi về baseline cho test cần thiết.")
print("- Nếu active_target là E4B: đây là hướng mạnh nhất còn có cửa trên T4.")
print("- Nếu fallback sang E2B: E4B không fit môi trường, dùng E2B lọc hàng loạt rồi E4B/A100 kiểm lại case khó.")
print("- Nếu tok/s vẫn thấp dù VRAM thừa: cổ chai là T4 + decoding/quant kernel, không phải thiếu VRAM.")
print_gpu("GPU CUỐI CELL")


0) CÀI / CẬP NHẬT THƯ VIỆN
1) GPU TRƯỚC KHI LOAD
Thu May 21 04:56:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


3) CẤU HÌNH
Target ưu tiên: google/gemma-4-E4B-it
Assistant ưu tiên: google/gemma-4-E4B-it-assistant
Fallback: google/gemma-4-E2B-it
dtype: torch.float16
TARGET_INT8_THRESHOLD: 0.0
MTP num_assistant_tokens: 8
MTP schedule: constant
MTP threshold: 0.0
RUN_MTP_SWEEP: True
MTP_SWEEP_TOKENS: [4, 8, 12, 16, 24]
RUN_BASELINE_COMPARE: False


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

OK: assistant 4-bit
OK: MTP strict n=8 schedule=constant threshold=0.0

assistant device map:
  cuda:0: 0.073B params
  total visible: 0.073B params
  OK: không thấy parameter trên CPU.
GPU SAU KHI LOAD ASSISTANT
Thu May 21 05:01:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0  

[transformers] An assistant model is provided, using a dynamic cache instead of a cache of type='static'.


input_tokens: 59
output_tokens: 12
seconds: 1.901
tokens_per_second: 6.311
used_static_cache: False
used_mtp: True
preview:
{"ok":true,"edge":"Tensor rỗng"}

FAST MODE: bỏ qua baseline_dfuzz. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.

--------------------------------------------------------------------------------------------------------------
TEST: mtp_dfuzz
--------------------------------------------------------------------------------------------------------------
active_target: google/gemma-4-E4B-it
target_mode: 8bit
active_assistant: google/gemma-4-E4B-it-assistant
assistant_mode: 4bit_nf4
prompt_chars: 800
prompt_tokens: 319
max_new_tokens: 180
use_mtp_requested: True
mtp_num_assistant_tokens: 8
mtp_schedule: constant
mtp_threshold: 0.0
input_tokens: 319
output_tokens: 173
seconds: 9.573
tokens_per_second: 18.072
used_static_cache: False
used_mtp: True
preview:
{"api": "upsample_bicubic2d", "params": [{"n": "input", "t": "Tensor"}, {"n": "output_size", "t": "List"}, {"n": "a

In [ ]:
# ============================================================
# GEMMA 4 E4B-it + MTP ASSISTANT OPTIMIZED TEST FOR COLAB T4
# Một cell duy nhất:
# cài -> login HF -> load E4B target -> load assistant -> benchmark baseline vs MTP -> test DFUZZ JSON
# ============================================================

import os, sys, gc, re, json, time, subprocess, traceback
from importlib.metadata import PackageNotFoundError, version

print("=" * 110)
print("0) CÀI / CẬP NHẬT THƯ VIỆN")
print("=" * 110)

PIP_PACKAGES = [
    "transformers>=5.5.0",
    "accelerate>=1.8.0",
    "bitsandbytes>=0.49.0",
    "huggingface_hub>=0.36.0",
    "hf_xet>=1.2.0",
    "sentencepiece",
    "protobuf<6,>=5.29.1",
]

os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
if os.path.exists("/content"):
    os.environ.setdefault("HF_HOME", "/content/hf_cache")

def version_tuple(text):
    nums = [int(x) for x in re.findall(r"\d+", text)[:3]]
    return tuple((nums + [0, 0, 0])[:3])

def installed_between(pkg, min_version=None, max_exclusive=None):
    try:
        current = version(pkg)
    except PackageNotFoundError:
        return False
    cur = version_tuple(current)
    if min_version is not None and cur < version_tuple(min_version):
        return False
    if max_exclusive is not None and cur >= version_tuple(max_exclusive):
        return False
    return True

def should_install_deps():
    mode = os.environ.get("INSTALL_DEPS", "auto").strip().lower()
    if mode in {"0", "false", "no", "skip"}:
        return False
    if mode in {"1", "true", "yes", "force"}:
        return True
    checks = [
        installed_between("transformers", "5.5.0"),
        installed_between("accelerate", "1.8.0"),
        installed_between("bitsandbytes", "0.49.0"),
        installed_between("huggingface_hub", "0.36.0"),
        installed_between("hf_xet", "1.2.0"),
        installed_between("sentencepiece"),
        installed_between("protobuf", "5.29.1", "6.0.0"),
    ]
    return not all(checks)

def env_bool(name, default):
    raw = os.environ.get(name, "1" if default else "0").strip().lower()
    return raw not in {"0", "false", "no", "skip"}

def env_float(name, default):
    raw = os.environ.get(name, str(default)).strip()
    try:
        return float(raw)
    except Exception:
        print(f"{name} không hợp lệ: {raw!r}; dùng default {default}.")
        return float(default)

def env_int_list(name, default):
    raw = os.environ.get(name, default).strip()
    vals = []
    for part in raw.split(","):
        part = part.strip()
        if not part:
            continue
        try:
            vals.append(int(part))
        except Exception:
            print(f"Bỏ qua {name} item không hợp lệ: {part!r}")
    return vals or [int(x.strip()) for x in default.split(",") if x.strip()]

def parse_mtp_configs(text):
    configs = []
    for item in text.split(","):
        item = item.strip()
        if not item:
            continue
        try:
            schedule, n, threshold = [x.strip() for x in item.split(":")]
            configs.append({
                "schedule": schedule,
                "tokens": int(n),
                "threshold": float(threshold),
            })
        except Exception:
            print(f"Bỏ qua MTP config không hợp lệ: {item!r}; format đúng: schedule:n:threshold")
    return configs

def env_mtp_configs(name, default):
    raw = os.environ.get(name, default).strip()
    configs = parse_mtp_configs(raw)
    if configs:
        return configs
    return parse_mtp_configs(default)

if should_install_deps():
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-U", *PIP_PACKAGES])
else:
    print("Bỏ qua pip install vì dependency đã đủ hoặc INSTALL_DEPS=0.")

import torch
from huggingface_hub import login, notebook_login
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# ------------------------------------------------------------
# 1) TIỆN ÍCH
# ------------------------------------------------------------
def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True)
    except Exception as e:
        return f"CMD ERROR: {repr(e)}"

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def print_gpu(title="GPU"):
    print("=" * 110)
    print(title)
    print("=" * 110)
    print(run_cmd("nvidia-smi"))
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("Capability:", torch.cuda.get_device_capability(0))
        try:
            print("BF16 supported:", torch.cuda.is_bf16_supported())
        except Exception as e:
            print("BF16 supported check error:", repr(e))

def first_model_device(m):
    for p in m.parameters():
        return p.device
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def count_devices(m, name="model"):
    from collections import Counter
    c = Counter()
    total = 0
    try:
        for _, p in m.named_parameters():
            c[str(p.device)] += p.numel()
            total += p.numel()
        print(f"\n{name} device map:")
        for dev, n in c.items():
            print(f"  {dev}: {n/1e9:.3f}B params")
        print(f"  total visible: {total/1e9:.3f}B params")
        if any("cpu" in d.lower() for d in c):
            print("  CẢNH BÁO: Có parameter trên CPU -> sẽ rất chậm.")
        else:
            print("  OK: không thấy parameter trên CPU.")
    except Exception as e:
        print(f"Không đếm được device map cho {name}:", repr(e))

print_gpu("1) GPU TRƯỚC KHI LOAD")

# ------------------------------------------------------------
# 2) TỐI ƯU CƠ BẢN CHO T4
# ------------------------------------------------------------
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

cleanup_cuda()

# ------------------------------------------------------------
# 3) LOGIN HUGGING FACE
# ------------------------------------------------------------
print("=" * 110)
print("2) HUGGING FACE LOGIN")
print("=" * 110)

hf_token = os.environ.get("HF_TOKEN", None)

try:
    from google.colab import userdata
    try:
        hf_token = hf_token or userdata.get("HF_TOKEN")
    except Exception:
        pass
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    print("OK: đã login bằng HF_TOKEN.")
else:
    print("Không thấy HF_TOKEN trong env/Colab secrets.")
    print("Nếu tải model bị 403/rate limit, hãy login ở widget hiện ra.")
    try:
        notebook_login()
    except Exception as e:
        print("Bỏ qua notebook_login hoặc lỗi login:", repr(e))

# ------------------------------------------------------------
# 4) CHỌN MODEL
# ------------------------------------------------------------
# Mạnh nhất còn có cửa chạy T4:
TARGET_MODEL_ID = "google/gemma-4-E4B-it"
ASSISTANT_MODEL_ID = "google/gemma-4-E4B-it-assistant"

# Fallback nếu E4B + assistant không fit:
FALLBACK_TARGET_MODEL_ID = "google/gemma-4-E2B-it"
FALLBACK_ASSISTANT_MODEL_ID = "google/gemma-4-E2B-it-assistant"

# T4: FP16 thực dụng hơn BF16.
DTYPE = torch.float16
TARGET_INT8_THRESHOLD = env_float("TARGET_INT8_THRESHOLD", 0.0)
MTP_NUM_ASSISTANT_TOKENS = int(os.environ.get("MTP_NUM_ASSISTANT_TOKENS", "4"))
MTP_ASSISTANT_SCHEDULE = "constant"
MTP_CONFIDENCE_THRESHOLD = 0.0
RUN_DFUZZ_MTP_SWEEP = env_bool("RUN_DFUZZ_MTP_SWEEP", True)
MTP_DFUZZ_SWEEP_CONFIGS = env_mtp_configs(
    "MTP_DFUZZ_SWEEP_CONFIGS",
    "constant:2:0.0,constant:3:0.0,constant:4:0.0,constant:6:0.0,constant:8:0.0,heuristic_transient:4:0.2,heuristic_transient:8:0.2",
)
RUN_MTP_SWEEP = env_bool("RUN_MTP_SWEEP", True)
MTP_SWEEP_TOKENS = env_int_list("MTP_SWEEP_TOKENS", "2,3,4,6,8,12")
RUN_BASELINE_COMPARE = False
RUN_WARMUP = True
RUN_FULL_DFUZZ = True
RUN_SHORT_JSON = True

print("=" * 110)
print("3) CẤU HÌNH")
print("=" * 110)
print("Target ưu tiên:", TARGET_MODEL_ID)
print("Assistant ưu tiên:", ASSISTANT_MODEL_ID)
print("Fallback:", FALLBACK_TARGET_MODEL_ID)
print("dtype:", DTYPE)
print("HF_XET_HIGH_PERFORMANCE:", os.environ.get("HF_XET_HIGH_PERFORMANCE"))
print("HF_HOME:", os.environ.get("HF_HOME"))
print("TARGET_INT8_THRESHOLD:", TARGET_INT8_THRESHOLD)
print("MTP num_assistant_tokens:", MTP_NUM_ASSISTANT_TOKENS)
print("MTP schedule:", MTP_ASSISTANT_SCHEDULE)
print("MTP threshold:", MTP_CONFIDENCE_THRESHOLD)
print("RUN_DFUZZ_MTP_SWEEP:", RUN_DFUZZ_MTP_SWEEP)
print("MTP_DFUZZ_SWEEP_CONFIGS:", MTP_DFUZZ_SWEEP_CONFIGS)
print("RUN_MTP_SWEEP:", RUN_MTP_SWEEP)
print("MTP_SWEEP_TOKENS:", MTP_SWEEP_TOKENS)
print("RUN_BASELINE_COMPARE:", RUN_BASELINE_COMPARE)

# ------------------------------------------------------------
# 5) LOAD PROCESSOR
# ------------------------------------------------------------
def load_processor(model_id):
    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
    )
    tokenizer = getattr(processor, "tokenizer", processor)
    try:
        tokenizer.padding_side = "left"
    except Exception:
        pass
    try:
        if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
    except Exception:
        pass
    return processor

# ------------------------------------------------------------
# 6) LOAD MODEL TARGET / ASSISTANT
# ------------------------------------------------------------
def load_target_8bit(model_id):
    q = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=TARGET_INT8_THRESHOLD,
        llm_int8_enable_fp32_cpu_offload=False,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_target_4bit(model_id):
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_assistant_4bit(model_id):
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_assistant_fp16(model_id):
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def configure_mtp_assistant(m, num_tokens=None, schedule=None, threshold=None):
    if m is None:
        return False
    cfg = getattr(m, "generation_config", None)
    if cfg is None:
        return False
    cfg.num_assistant_tokens = MTP_NUM_ASSISTANT_TOKENS if num_tokens is None else int(num_tokens)
    cfg.num_assistant_tokens_schedule = MTP_ASSISTANT_SCHEDULE if schedule is None else schedule
    cfg.assistant_confidence_threshold = MTP_CONFIDENCE_THRESHOLD if threshold is None else float(threshold)
    return True

def load_stack(target_id, assistant_id):
    """
    Chiến lược:
    1. Target E4B 8-bit vì chất lượng tốt hơn 4-bit.
    2. Assistant 4-bit để cố fit VRAM T4.
    3. Nếu OOM: target E4B 4-bit + assistant 4-bit.
    4. Nếu vẫn fail: fallback E2B 8-bit + assistant 4-bit.
    5. Nếu assistant fail: vẫn giữ target để benchmark baseline.
    """
    global processor, model, assistant_model, active_target_id, active_assistant_id, target_mode, assistant_mode

    processor = load_processor(target_id)
    active_target_id = target_id
    active_assistant_id = assistant_id
    model = None
    assistant_model = None
    target_mode = None
    assistant_mode = None

    print("=" * 110)
    print("4) LOAD TARGET MODEL")
    print("=" * 110)

    try:
        print("Thử target 8-bit:", target_id)
        model = load_target_8bit(target_id)
        target_mode = "8bit"
        print("OK: target 8-bit")
    except Exception as e1:
        print("FAIL target 8-bit:", repr(e1))
        traceback.print_exc(limit=1)
        cleanup_cuda()
        print("Thử target 4-bit:", target_id)
        model = load_target_4bit(target_id)
        target_mode = "4bit_nf4"
        print("OK: target 4-bit")

    count_devices(model, "target")
    print_gpu("GPU SAU KHI LOAD TARGET")

    print("=" * 110)
    print("5) LOAD ASSISTANT MODEL CHO MTP")
    print("=" * 110)

    # Trên T4 ưu tiên assistant 4-bit.
    try:
        print("Thử assistant 4-bit:", assistant_id)
        assistant_model = load_assistant_4bit(assistant_id)
        assistant_mode = "4bit_nf4"
        print("OK: assistant 4-bit")
        if configure_mtp_assistant(assistant_model):
            print(
                "OK: MTP strict",
                f"n={MTP_NUM_ASSISTANT_TOKENS}",
                f"schedule={MTP_ASSISTANT_SCHEDULE}",
                f"threshold={MTP_CONFIDENCE_THRESHOLD}",
            )
    except Exception as e:
        print("FAIL assistant 4-bit:", repr(e))
        traceback.print_exc(limit=1)
        cleanup_cuda()
        assistant_model = None
        assistant_mode = None
        print("Không có assistant -> vẫn benchmark baseline target.")

    if assistant_model is not None:
        count_devices(assistant_model, "assistant")
    print_gpu("GPU SAU KHI LOAD ASSISTANT")

try:
    load_stack(TARGET_MODEL_ID, ASSISTANT_MODEL_ID)
except Exception as e:
    print("\n" + "!" * 110)
    print("E4B STACK FAIL. FALLBACK SANG E2B STACK.")
    print("Lỗi:", repr(e))
    print("!" * 110)
    traceback.print_exc(limit=2)
    cleanup_cuda()
    load_stack(FALLBACK_TARGET_MODEL_ID, FALLBACK_ASSISTANT_MODEL_ID)

# ------------------------------------------------------------
# 7) PROMPT DFUZZ GỌN
# ------------------------------------------------------------
SYSTEM_PROMPT = (
    "Bạn là bộ phân tích mã C++ cho fuzzing API học sâu. "
    "Chỉ trả JSON hợp lệ. Không markdown. Không giải thích."
)

SCHEMA_MIN = {
    "api": "string",
    "params": [{"n": "string", "t": "Tensor|Int|Bool|Str|Float|Scalar|List|Optional|Unknown"}],
    "cases": [{"cond": "string", "bad": "string", "p": ["string"], "t": ["string"], "conf": 0.0}]
}

DFUZZ_PROMPT = f"""
Phân tích check C++ và rút ca biên vi phạm.

Luật:
- Chỉ dùng type: Tensor, Int, Bool, Str, Float, Scalar, List, Optional, Unknown.
- JSON phải đúng schema.
- Viết cực ngắn.

Schema:
{json.dumps(SCHEMA_MIN, ensure_ascii=False)}

Signature:
TORCH_META_FUNC(upsample_bicubic2d)(
    const Tensor& input,
    IntArrayRef output_size,
    bool align_corners,
    c10::optional<double> scales_h,
    c10::optional<double> scales_w)

Check:
TORCH_CHECK(
    input.numel() != 0 ||
    c10::multiply_integers(input.sizes().begin() + 1, input.sizes().end()),
    "Non-empty 4D data tensor expected but got a tensor with sizes ",
    input.sizes()
);
""".strip()

SHORT_PROMPT = 'Trả JSON hợp lệ: {"ok":true,"edge":"Tensor rỗng"}'

# ------------------------------------------------------------
# 8) TOKENIZE / GENERATE / BENCHMARK
# ------------------------------------------------------------
def make_inputs(user_prompt, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    dev = first_model_device(model)
    inputs = {
        k: v.to(dev) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }
    return inputs

def extract_first_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    if start < 0:
        raise ValueError("Không thấy JSON object")

    depth = 0
    in_str = False
    esc = False

    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]

    raise ValueError("JSON chưa đóng ngoặc")

def parse_json_loose(text):
    return json.loads(extract_first_json_object(text))

def generate_once(
    prompt,
    max_new_tokens=160,
    use_mtp=False,
    use_static_cache=True,
    mtp_tokens=None,
    mtp_schedule=None,
    mtp_threshold=None,
):
    inputs = make_inputs(prompt)
    input_tokens = int(inputs["input_ids"].shape[-1])

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    if use_static_cache:
        gen_kwargs["cache_implementation"] = "static"

    if use_mtp and assistant_model is not None:
        configure_mtp_assistant(assistant_model, mtp_tokens, mtp_schedule, mtp_threshold)
        gen_kwargs["assistant_model"] = assistant_model

    mtp_cfg = getattr(assistant_model, "generation_config", None) if use_mtp and assistant_model is not None else None
    actual_mtp_tokens = getattr(mtp_cfg, "num_assistant_tokens", None) if mtp_cfg is not None else None
    actual_mtp_schedule = getattr(mtp_cfg, "num_assistant_tokens_schedule", None) if mtp_cfg is not None else None
    actual_mtp_threshold = getattr(mtp_cfg, "assistant_confidence_threshold", None) if mtp_cfg is not None else None

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    used_mtp = bool(use_mtp and assistant_model is not None)
    # Transformers currently switches assisted decoding to dynamic cache even if static was requested.
    used_static = bool(use_static_cache and not used_mtp)

    try:
        with torch.inference_mode():
            out = model.generate(**gen_kwargs)
    except Exception as e:
        msg = str(e).lower()
        # Một số backend/model chưa nhận static cache hoặc assistant + static cùng lúc.
        if "cache_implementation" in msg or "static" in msg:
            print("Static cache lỗi -> fallback dynamic cache:", repr(e))
            gen_kwargs.pop("cache_implementation", None)
            used_static = False
            with torch.inference_mode():
                out = model.generate(**gen_kwargs)
        elif used_mtp:
            print("MTP lỗi -> fallback baseline không assistant:", repr(e))
            gen_kwargs.pop("assistant_model", None)
            used_mtp = False
            with torch.inference_mode():
                out = model.generate(**gen_kwargs)
        else:
            raise

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t1 = time.perf_counter()

    output_tokens = int(out[0].shape[-1] - input_tokens)
    elapsed = t1 - t0
    tok_s = output_tokens / elapsed if elapsed > 0 else 0.0
    decoded = processor.decode(out[0][input_tokens:], skip_special_tokens=True)

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "seconds": elapsed,
        "tok_s": tok_s,
        "used_static_cache": used_static,
        "used_mtp": used_mtp,
        "mtp_tokens": actual_mtp_tokens,
        "mtp_schedule": actual_mtp_schedule,
        "mtp_threshold": actual_mtp_threshold,
        "text": decoded,
    }

def bench(
    name,
    prompt,
    max_new_tokens=160,
    use_mtp=False,
    parse=False,
    mtp_tokens=None,
    mtp_schedule=None,
    mtp_threshold=None,
):
    print("\n" + "-" * 110)
    print("TEST:", name)
    print("-" * 110)

    inp = make_inputs(prompt)
    print("active_target:", active_target_id)
    print("target_mode:", target_mode)
    print("active_assistant:", active_assistant_id if assistant_model is not None else None)
    print("assistant_mode:", assistant_mode)
    print("prompt_chars:", len(prompt))
    print("prompt_tokens:", int(inp["input_ids"].shape[-1]))
    print("max_new_tokens:", max_new_tokens)
    print("use_mtp_requested:", use_mtp)
    if use_mtp and assistant_model is not None:
        configure_mtp_assistant(assistant_model, mtp_tokens, mtp_schedule, mtp_threshold)
        cfg = getattr(assistant_model, "generation_config", None)
        if cfg is not None:
            print("mtp_num_assistant_tokens:", getattr(cfg, "num_assistant_tokens", None))
            print("mtp_schedule:", getattr(cfg, "num_assistant_tokens_schedule", None))
            print("mtp_threshold:", getattr(cfg, "assistant_confidence_threshold", None))

    r = generate_once(
        prompt,
        max_new_tokens=max_new_tokens,
        use_mtp=use_mtp,
        use_static_cache=True,
        mtp_tokens=mtp_tokens,
        mtp_schedule=mtp_schedule,
        mtp_threshold=mtp_threshold,
    )

    print("input_tokens:", r["input_tokens"])
    print("output_tokens:", r["output_tokens"])
    print("seconds:", round(r["seconds"], 3))
    print("tokens_per_second:", round(r["tok_s"], 3))
    print("used_static_cache:", r["used_static_cache"])
    print("used_mtp:", r["used_mtp"])
    if r["used_mtp"]:
        print("actual_mtp_tokens:", r["mtp_tokens"])
        print("actual_mtp_schedule:", r["mtp_schedule"])
        print("actual_mtp_threshold:", r["mtp_threshold"])
    print("preview:")
    print(r["text"][:1600])

    if parse:
        try:
            obj = parse_json_loose(r["text"])
            print("\nJSON PARSE: OK")
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:2500])
        except Exception as e:
            print("\nJSON PARSE: LỖI:", repr(e))

    return r

# ------------------------------------------------------------
# 9) CHẠY BENCHMARK
# ------------------------------------------------------------
print("=" * 110)
print("6) BENCHMARK FAST MTP / BASELINE OPTIONAL")
print("=" * 110)

results = []

fast_use_mtp = assistant_model is not None

if RUN_WARMUP:
    warmup_name = "warmup_mtp_short" if fast_use_mtp else "warmup_baseline_short"
    results.append((warmup_name, bench(
        warmup_name,
        SHORT_PROMPT,
        max_new_tokens=48,
        use_mtp=fast_use_mtp,
        parse=False,
    )))

if RUN_BASELINE_COMPARE:
    results.append(("baseline_dfuzz", bench(
        "baseline_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=False,
        parse=True,
    )))
else:
    print("\nFAST MODE: bỏ qua baseline_dfuzz. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.")

if RUN_FULL_DFUZZ and assistant_model is not None and RUN_DFUZZ_MTP_SWEEP:
    seen_mtp_configs = set()
    for cfg in MTP_DFUZZ_SWEEP_CONFIGS:
        key = (cfg["schedule"], cfg["tokens"], cfg["threshold"])
        if key in seen_mtp_configs:
            continue
        seen_mtp_configs.add(key)
        name = f"mtp_dfuzz_{cfg['schedule']}_n{cfg['tokens']}_t{cfg['threshold']:g}"
        results.append((name, bench(
            name,
            DFUZZ_PROMPT,
            max_new_tokens=180,
            use_mtp=True,
            parse=True,
            mtp_tokens=cfg["tokens"],
            mtp_schedule=cfg["schedule"],
            mtp_threshold=cfg["threshold"],
        )))
elif RUN_FULL_DFUZZ and assistant_model is not None:
    results.append(("mtp_dfuzz", bench(
        "mtp_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=True,
        parse=True,
    )))
elif RUN_FULL_DFUZZ and not RUN_BASELINE_COMPARE:
    results.append(("baseline_dfuzz", bench(
        "baseline_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=False,
        parse=True,
    )))
elif RUN_FULL_DFUZZ:
    print("\nKhông có assistant_model -> bỏ qua MTP benchmark.")

# Một lượt output ngắn hơn, vì DFUZZ thực tế nên ép JSON ngắn.
SHORT_JSON_DFUZZ_PROMPT = """
Rút ca biên từ check C++ sau. Chỉ trả JSON một dòng.
Schema: {"api":"str","bad":"str","p":["str"],"t":["str"],"conf":0.0}
Signature: upsample_bicubic2d(Tensor input, List output_size, Bool align_corners, Optional scales_h, Optional scales_w)
Check: input.numel()!=0 || product(input.sizes()[1:]) != 0
""".strip()

if RUN_BASELINE_COMPARE:
    results.append(("baseline_short_json", bench(
        "baseline_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=False,
        parse=True,
    )))
else:
    print("\nFAST MODE: bỏ qua baseline_short_json. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.")

if RUN_SHORT_JSON and assistant_model is not None and RUN_MTP_SWEEP:
    seen_mtp_tokens = set()
    for n in MTP_SWEEP_TOKENS:
        if n in seen_mtp_tokens:
            continue
        seen_mtp_tokens.add(n)
        name = f"mtp_short_json_n{n}"
        results.append((name, bench(
            name,
            SHORT_JSON_DFUZZ_PROMPT,
            max_new_tokens=96,
            use_mtp=True,
            parse=True,
            mtp_tokens=n,
        )))
elif RUN_SHORT_JSON and assistant_model is not None:
    results.append(("mtp_short_json", bench(
        "mtp_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=True,
        parse=True,
    )))
elif RUN_SHORT_JSON and not RUN_BASELINE_COMPARE:
    results.append(("baseline_short_json", bench(
        "baseline_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=False,
        parse=True,
    )))

# ------------------------------------------------------------
# 10) TỔNG KẾT
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("7) TỔNG KẾT")
print("=" * 110)

summary = []
for name, r in results:
    summary.append({
        "name": name,
        "in_tok": r["input_tokens"],
        "out_tok": r["output_tokens"],
        "seconds": round(r["seconds"], 3),
        "tok_s": round(r["tok_s"], 3),
        "static": r["used_static_cache"],
        "mtp": r["used_mtp"],
        "mtp_n": r.get("mtp_tokens"),
        "mtp_schedule": r.get("mtp_schedule"),
        "mtp_threshold": r.get("mtp_threshold"),
    })

print(json.dumps(summary, ensure_ascii=False, indent=2))

base = next((r for n, r in results if n == "baseline_dfuzz"), None)
mtp_candidates = [(n, r) for n, r in results if n.startswith("mtp_dfuzz")]
mtp_name, mtp = max(mtp_candidates, key=lambda x: x[1]["tok_s"]) if mtp_candidates else (None, None)

if base and mtp:
    speedup = mtp["tok_s"] / base["tok_s"] if base["tok_s"] > 0 else 0
    time_ratio = base["seconds"] / mtp["seconds"] if mtp["seconds"] > 0 else 0
    print("\nSo sánh DFUZZ:")
    print("baseline tok/s:", round(base["tok_s"], 3))
    print("MTP tốt nhất:", mtp_name)
    print("MTP tok/s:", round(mtp["tok_s"], 3))
    print("tok/s speedup:", round(speedup, 3), "x")
    print("wall-time speedup:", round(time_ratio, 3), "x")

meaningful_summary = [x for x in summary if not x["name"].startswith("warmup")]
if meaningful_summary:
    fastest_tok = max(meaningful_summary, key=lambda x: x["tok_s"])
    fastest_wall = min(meaningful_summary, key=lambda x: x["seconds"])
    print("\nNhanh nhất:")
    print("tok/s:", fastest_tok["name"], fastest_tok["tok_s"], "tok/s")
    print("wall-time:", fastest_wall["name"], fastest_wall["seconds"], "s")

print("\nKết luận đọc nhanh:")
print("- Mặc định fast mode: target hiện tại + assistant_model + MTP strict n=4/constant/0.0.")
print("- RUN_DFUZZ_MTP_SWEEP=True sẽ tự thử các cấu hình MTP đáng nghi nhất cho DFUZZ.")
print("- Muốn so baseline lại: đặt RUN_BASELINE_COMPARE=True.")
print("- Nếu MTP OOM/lỗi: script tự rơi về baseline cho test cần thiết.")
print("- Nếu active_target là E4B: đây là hướng mạnh nhất còn có cửa trên T4.")
print("- Nếu fallback sang E2B: E4B không fit môi trường, dùng E2B lọc hàng loạt rồi E4B/A100 kiểm lại case khó.")
print("- Nếu tok/s vẫn thấp dù VRAM thừa: cổ chai là T4 + decoding/quant kernel, không phải thiếu VRAM.")
print_gpu("GPU CUỐI CELL")


0) CÀI / CẬP NHẬT THƯ VIỆN
1) GPU TRƯỚC KHI LOAD
Thu May 21 14:34:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


3) CẤU HÌNH
Target ưu tiên: google/gemma-4-E4B-it
Assistant ưu tiên: google/gemma-4-E4B-it-assistant
Fallback: google/gemma-4-E2B-it
dtype: torch.float16
HF_XET_HIGH_PERFORMANCE: 1
HF_HOME: /content/hf_cache
TARGET_INT8_THRESHOLD: 0.0
MTP num_assistant_tokens: 4
MTP schedule: constant
MTP threshold: 0.0
RUN_DFUZZ_MTP_SWEEP: True
MTP_DFUZZ_SWEEP_CONFIGS: [{'schedule': 'constant', 'tokens': 2, 'threshold': 0.0}, {'schedule': 'constant', 'tokens': 3, 'threshold': 0.0}, {'schedule': 'constant', 'tokens': 4, 'threshold': 0.0}, {'schedule': 'constant', 'tokens': 6, 'threshold': 0.0}, {'schedule': 'constant', 'tokens': 8, 'threshold': 0.0}, {'schedule': 'heuristic_transient', 'tokens': 4, 'threshold': 0.2}, {'schedule': 'heuristic_transient', 'tokens': 8, 'threshold': 0.2}]
RUN_MTP_SWEEP: True
MTP_SWEEP_TOKENS: [2, 3, 4, 6, 8, 12]
RUN_BASELINE_COMPARE: False


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

4) LOAD TARGET MODEL
Thử target 8-bit: google/gemma-4-E4B-it


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

In [ ]:
# ============================================================
# GEMMA 4 E4B-it + MTP ASSISTANT OPTIMIZED TEST FOR COLAB T4
# Một cell duy nhất:
# cài -> login HF -> load E4B target -> load assistant -> benchmark baseline vs MTP -> test DFUZZ JSON
# ============================================================

import os, sys, gc, re, json, time, subprocess, traceback
from importlib.metadata import PackageNotFoundError, version

print("=" * 110)
print("0) CÀI / CẬP NHẬT THƯ VIỆN")
print("=" * 110)

PIP_PACKAGES = [
    "transformers>=5.5.0",
    "accelerate>=1.8.0",
    "bitsandbytes>=0.49.0",
    "huggingface_hub>=0.36.0",
    "hf_xet>=1.2.0",
    "sentencepiece",
    "protobuf<6,>=5.29.1",
]

if os.path.exists("/content"):
    os.environ.setdefault("HF_HOME", "/content/hf_cache")
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# Colab free/T4 hay chết RAM khi hf-xet high-performance cố saturate CPU/disk/network.
# Mặc định dùng Xet chế độ an toàn; muốn ép nhanh thì đặt HF_XET_HIGH_PERFORMANCE=1 trước khi chạy cell.
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "0")
os.environ.setdefault("HF_XET_NUM_CONCURRENT_RANGE_GETS", "4")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
os.environ.setdefault("HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY", "1")
if os.environ.get("HF_HUB_DISABLE_XET", "").strip().lower() in {"1", "true", "yes"}:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

def version_tuple(text):
    nums = [int(x) for x in re.findall(r"\d+", text)[:3]]
    return tuple((nums + [0, 0, 0])[:3])

def installed_between(pkg, min_version=None, max_exclusive=None):
    try:
        current = version(pkg)
    except PackageNotFoundError:
        return False
    cur = version_tuple(current)
    if min_version is not None and cur < version_tuple(min_version):
        return False
    if max_exclusive is not None and cur >= version_tuple(max_exclusive):
        return False
    return True

def should_install_deps():
    mode = os.environ.get("INSTALL_DEPS", "auto").strip().lower()
    if mode in {"0", "false", "no", "skip"}:
        return False
    if mode in {"1", "true", "yes", "force"}:
        return True
    checks = [
        installed_between("transformers", "5.5.0"),
        installed_between("accelerate", "1.8.0"),
        installed_between("bitsandbytes", "0.49.0"),
        installed_between("huggingface_hub", "0.36.0"),
        installed_between("hf_xet", "1.2.0"),
        installed_between("sentencepiece"),
        installed_between("protobuf", "5.29.1", "6.0.0"),
    ]
    return not all(checks)

def env_bool(name, default):
    raw = os.environ.get(name, "1" if default else "0").strip().lower()
    return raw not in {"0", "false", "no", "skip"}

def env_float(name, default):
    raw = os.environ.get(name, str(default)).strip()
    try:
        return float(raw)
    except Exception:
        print(f"{name} không hợp lệ: {raw!r}; dùng default {default}.")
        return float(default)

def env_int_list(name, default):
    raw = os.environ.get(name, default).strip()
    vals = []
    for part in raw.split(","):
        part = part.strip()
        if not part:
            continue
        try:
            vals.append(int(part))
        except Exception:
            print(f"Bỏ qua {name} item không hợp lệ: {part!r}")
    return vals or [int(x.strip()) for x in default.split(",") if x.strip()]

def parse_mtp_configs(text):
    configs = []
    for item in text.split(","):
        item = item.strip()
        if not item:
            continue
        try:
            schedule, n, threshold = [x.strip() for x in item.split(":")]
            configs.append({
                "schedule": schedule,
                "tokens": int(n),
                "threshold": float(threshold),
            })
        except Exception:
            print(f"Bỏ qua MTP config không hợp lệ: {item!r}; format đúng: schedule:n:threshold")
    return configs

def env_mtp_configs(name, default):
    raw = os.environ.get(name, default).strip()
    configs = parse_mtp_configs(raw)
    if configs:
        return configs
    return parse_mtp_configs(default)

if should_install_deps():
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-U", *PIP_PACKAGES])
else:
    print("Bỏ qua pip install vì dependency đã đủ hoặc INSTALL_DEPS=0.")

import torch
from huggingface_hub import login, notebook_login
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# ------------------------------------------------------------
# 1) TIỆN ÍCH
# ------------------------------------------------------------
def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True)
    except Exception as e:
        return f"CMD ERROR: {repr(e)}"

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def print_gpu(title="GPU"):
    print("=" * 110)
    print(title)
    print("=" * 110)
    print(run_cmd("nvidia-smi"))
    if os.path.exists("/content"):
        print("RAM:")
        print(run_cmd("free -h"))
        print("DISK:")
        print(run_cmd("df -h /content /content/hf_cache 2>/dev/null"))
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("Capability:", torch.cuda.get_device_capability(0))
        try:
            print("BF16 supported:", torch.cuda.is_bf16_supported())
        except Exception as e:
            print("BF16 supported check error:", repr(e))

def first_model_device(m):
    for p in m.parameters():
        return p.device
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def count_devices(m, name="model"):
    from collections import Counter
    c = Counter()
    total = 0
    try:
        for _, p in m.named_parameters():
            c[str(p.device)] += p.numel()
            total += p.numel()
        print(f"\n{name} device map:")
        for dev, n in c.items():
            print(f"  {dev}: {n/1e9:.3f}B params")
        print(f"  total visible: {total/1e9:.3f}B params")
        if any("cpu" in d.lower() for d in c):
            print("  CẢNH BÁO: Có parameter trên CPU -> sẽ rất chậm.")
        else:
            print("  OK: không thấy parameter trên CPU.")
    except Exception as e:
        print(f"Không đếm được device map cho {name}:", repr(e))

print_gpu("1) GPU TRƯỚC KHI LOAD")

# ------------------------------------------------------------
# 2) TỐI ƯU CƠ BẢN CHO T4
# ------------------------------------------------------------
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

cleanup_cuda()

# ------------------------------------------------------------
# 3) LOGIN HUGGING FACE
# ------------------------------------------------------------
print("=" * 110)
print("2) HUGGING FACE LOGIN")
print("=" * 110)

hf_token = os.environ.get("HF_TOKEN", None)

try:
    from google.colab import userdata
    try:
        hf_token = hf_token or userdata.get("HF_TOKEN")
    except Exception:
        pass
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    print("OK: đã login bằng HF_TOKEN.")
else:
    print("Không thấy HF_TOKEN trong env/Colab secrets.")
    print("Nếu tải model bị 403/rate limit, hãy login ở widget hiện ra.")
    try:
        notebook_login()
    except Exception as e:
        print("Bỏ qua notebook_login hoặc lỗi login:", repr(e))

# ------------------------------------------------------------
# 4) CHỌN MODEL
# ------------------------------------------------------------
# Mạnh nhất còn có cửa chạy T4:
TARGET_MODEL_ID = "google/gemma-4-E4B-it"
ASSISTANT_MODEL_ID = "google/gemma-4-E4B-it-assistant"

# Fallback nếu E4B + assistant không fit:
FALLBACK_TARGET_MODEL_ID = "google/gemma-4-E2B-it"
FALLBACK_ASSISTANT_MODEL_ID = "google/gemma-4-E2B-it-assistant"

# T4: FP16 thực dụng hơn BF16.
DTYPE = torch.float16
TARGET_INT8_THRESHOLD = env_float("TARGET_INT8_THRESHOLD", 0.0)
MTP_NUM_ASSISTANT_TOKENS = int(os.environ.get("MTP_NUM_ASSISTANT_TOKENS", "4"))
MTP_ASSISTANT_SCHEDULE = "constant"
MTP_CONFIDENCE_THRESHOLD = 0.0
RUN_DFUZZ_MTP_SWEEP = env_bool("RUN_DFUZZ_MTP_SWEEP", True)
MTP_DFUZZ_SWEEP_CONFIGS = env_mtp_configs(
    "MTP_DFUZZ_SWEEP_CONFIGS",
    "constant:2:0.0,constant:3:0.0,constant:4:0.0,constant:6:0.0,constant:8:0.0,heuristic_transient:4:0.2,heuristic_transient:8:0.2",
)
RUN_MTP_SWEEP = env_bool("RUN_MTP_SWEEP", True)
MTP_SWEEP_TOKENS = env_int_list("MTP_SWEEP_TOKENS", "2,3,4,6,8,12")
RUN_BASELINE_COMPARE = False
RUN_WARMUP = True
RUN_FULL_DFUZZ = True
RUN_SHORT_JSON = True

print("=" * 110)
print("3) CẤU HÌNH")
print("=" * 110)
print("Target ưu tiên:", TARGET_MODEL_ID)
print("Assistant ưu tiên:", ASSISTANT_MODEL_ID)
print("Fallback:", FALLBACK_TARGET_MODEL_ID)
print("dtype:", DTYPE)
print("HF_XET_HIGH_PERFORMANCE:", os.environ.get("HF_XET_HIGH_PERFORMANCE"))
print("HF_XET_NUM_CONCURRENT_RANGE_GETS:", os.environ.get("HF_XET_NUM_CONCURRENT_RANGE_GETS"))
print("HF_XET_CHUNK_CACHE_SIZE_BYTES:", os.environ.get("HF_XET_CHUNK_CACHE_SIZE_BYTES"))
print("HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY:", os.environ.get("HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY"))
print("HF_HUB_DISABLE_XET:", os.environ.get("HF_HUB_DISABLE_XET"))
print("HF_HOME:", os.environ.get("HF_HOME"))
print("TARGET_INT8_THRESHOLD:", TARGET_INT8_THRESHOLD)
print("MTP num_assistant_tokens:", MTP_NUM_ASSISTANT_TOKENS)
print("MTP schedule:", MTP_ASSISTANT_SCHEDULE)
print("MTP threshold:", MTP_CONFIDENCE_THRESHOLD)
print("RUN_DFUZZ_MTP_SWEEP:", RUN_DFUZZ_MTP_SWEEP)
print("MTP_DFUZZ_SWEEP_CONFIGS:", MTP_DFUZZ_SWEEP_CONFIGS)
print("RUN_MTP_SWEEP:", RUN_MTP_SWEEP)
print("MTP_SWEEP_TOKENS:", MTP_SWEEP_TOKENS)
print("RUN_BASELINE_COMPARE:", RUN_BASELINE_COMPARE)

# ------------------------------------------------------------
# 5) LOAD PROCESSOR
# ------------------------------------------------------------
def load_processor(model_id):
    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
    )
    tokenizer = getattr(processor, "tokenizer", processor)
    try:
        tokenizer.padding_side = "left"
    except Exception:
        pass
    try:
        if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
    except Exception:
        pass
    return processor

# ------------------------------------------------------------
# 6) LOAD MODEL TARGET / ASSISTANT
# ------------------------------------------------------------
def load_target_8bit(model_id):
    q = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=TARGET_INT8_THRESHOLD,
        llm_int8_enable_fp32_cpu_offload=False,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_target_4bit(model_id):
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_assistant_4bit(model_id):
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            quantization_config=q,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def load_assistant_fp16(model_id):
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            dtype=DTYPE,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=DTYPE,
            attn_implementation="sdpa",
            trust_remote_code=True,
        ).eval()

def configure_mtp_assistant(m, num_tokens=None, schedule=None, threshold=None):
    if m is None:
        return False
    cfg = getattr(m, "generation_config", None)
    if cfg is None:
        return False
    cfg.num_assistant_tokens = MTP_NUM_ASSISTANT_TOKENS if num_tokens is None else int(num_tokens)
    cfg.num_assistant_tokens_schedule = MTP_ASSISTANT_SCHEDULE if schedule is None else schedule
    cfg.assistant_confidence_threshold = MTP_CONFIDENCE_THRESHOLD if threshold is None else float(threshold)
    return True

def load_stack(target_id, assistant_id):
    """
    Chiến lược:
    1. Target E4B 8-bit vì chất lượng tốt hơn 4-bit.
    2. Assistant 4-bit để cố fit VRAM T4.
    3. Nếu OOM: target E4B 4-bit + assistant 4-bit.
    4. Nếu vẫn fail: fallback E2B 8-bit + assistant 4-bit.
    5. Nếu assistant fail: vẫn giữ target để benchmark baseline.
    """
    global processor, model, assistant_model, active_target_id, active_assistant_id, target_mode, assistant_mode

    processor = load_processor(target_id)
    active_target_id = target_id
    active_assistant_id = assistant_id
    model = None
    assistant_model = None
    target_mode = None
    assistant_mode = None

    print("=" * 110)
    print("4) LOAD TARGET MODEL")
    print("=" * 110)

    try:
        print("Thử target 8-bit:", target_id)
        model = load_target_8bit(target_id)
        target_mode = "8bit"
        print("OK: target 8-bit")
    except Exception as e1:
        print("FAIL target 8-bit:", repr(e1))
        traceback.print_exc(limit=1)
        cleanup_cuda()
        print("Thử target 4-bit:", target_id)
        model = load_target_4bit(target_id)
        target_mode = "4bit_nf4"
        print("OK: target 4-bit")

    count_devices(model, "target")
    print_gpu("GPU SAU KHI LOAD TARGET")

    print("=" * 110)
    print("5) LOAD ASSISTANT MODEL CHO MTP")
    print("=" * 110)

    # Trên T4 ưu tiên assistant 4-bit.
    try:
        print("Thử assistant 4-bit:", assistant_id)
        assistant_model = load_assistant_4bit(assistant_id)
        assistant_mode = "4bit_nf4"
        print("OK: assistant 4-bit")
        if configure_mtp_assistant(assistant_model):
            print(
                "OK: MTP strict",
                f"n={MTP_NUM_ASSISTANT_TOKENS}",
                f"schedule={MTP_ASSISTANT_SCHEDULE}",
                f"threshold={MTP_CONFIDENCE_THRESHOLD}",
            )
    except Exception as e:
        print("FAIL assistant 4-bit:", repr(e))
        traceback.print_exc(limit=1)
        cleanup_cuda()
        assistant_model = None
        assistant_mode = None
        print("Không có assistant -> vẫn benchmark baseline target.")

    if assistant_model is not None:
        count_devices(assistant_model, "assistant")
    print_gpu("GPU SAU KHI LOAD ASSISTANT")

try:
    load_stack(TARGET_MODEL_ID, ASSISTANT_MODEL_ID)
except Exception as e:
    print("\n" + "!" * 110)
    print("E4B STACK FAIL. FALLBACK SANG E2B STACK.")
    print("Lỗi:", repr(e))
    print("!" * 110)
    traceback.print_exc(limit=2)
    cleanup_cuda()
    load_stack(FALLBACK_TARGET_MODEL_ID, FALLBACK_ASSISTANT_MODEL_ID)

# ------------------------------------------------------------
# 7) PROMPT DFUZZ GỌN
# ------------------------------------------------------------
SYSTEM_PROMPT = (
    "Bạn là bộ phân tích mã C++ cho fuzzing API học sâu. "
    "Chỉ trả JSON hợp lệ. Không markdown. Không giải thích."
)

SCHEMA_MIN = {
    "api": "string",
    "params": [{"n": "string", "t": "Tensor|Int|Bool|Str|Float|Scalar|List|Optional|Unknown"}],
    "cases": [{"cond": "string", "bad": "string", "p": ["string"], "t": ["string"], "conf": 0.0}]
}

DFUZZ_PROMPT = f"""
Phân tích check C++ và rút ca biên vi phạm.

Luật:
- Chỉ dùng type: Tensor, Int, Bool, Str, Float, Scalar, List, Optional, Unknown.
- JSON phải đúng schema.
- Viết cực ngắn.

Schema:
{json.dumps(SCHEMA_MIN, ensure_ascii=False)}

Signature:
TORCH_META_FUNC(upsample_bicubic2d)(
    const Tensor& input,
    IntArrayRef output_size,
    bool align_corners,
    c10::optional<double> scales_h,
    c10::optional<double> scales_w)

Check:
TORCH_CHECK(
    input.numel() != 0 ||
    c10::multiply_integers(input.sizes().begin() + 1, input.sizes().end()),
    "Non-empty 4D data tensor expected but got a tensor with sizes ",
    input.sizes()
);
""".strip()

SHORT_PROMPT = 'Trả JSON hợp lệ: {"ok":true,"edge":"Tensor rỗng"}'

# ------------------------------------------------------------
# 8) TOKENIZE / GENERATE / BENCHMARK
# ------------------------------------------------------------
def make_inputs(user_prompt, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    dev = first_model_device(model)
    inputs = {
        k: v.to(dev) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }
    return inputs

def extract_first_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    if start < 0:
        raise ValueError("Không thấy JSON object")

    depth = 0
    in_str = False
    esc = False

    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]

    raise ValueError("JSON chưa đóng ngoặc")

def parse_json_loose(text):
    return json.loads(extract_first_json_object(text))

def generate_once(
    prompt,
    max_new_tokens=160,
    use_mtp=False,
    use_static_cache=True,
    mtp_tokens=None,
    mtp_schedule=None,
    mtp_threshold=None,
):
    inputs = make_inputs(prompt)
    input_tokens = int(inputs["input_ids"].shape[-1])

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    if use_static_cache:
        gen_kwargs["cache_implementation"] = "static"

    if use_mtp and assistant_model is not None:
        configure_mtp_assistant(assistant_model, mtp_tokens, mtp_schedule, mtp_threshold)
        gen_kwargs["assistant_model"] = assistant_model

    mtp_cfg = getattr(assistant_model, "generation_config", None) if use_mtp and assistant_model is not None else None
    actual_mtp_tokens = getattr(mtp_cfg, "num_assistant_tokens", None) if mtp_cfg is not None else None
    actual_mtp_schedule = getattr(mtp_cfg, "num_assistant_tokens_schedule", None) if mtp_cfg is not None else None
    actual_mtp_threshold = getattr(mtp_cfg, "assistant_confidence_threshold", None) if mtp_cfg is not None else None

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    used_mtp = bool(use_mtp and assistant_model is not None)
    # Transformers currently switches assisted decoding to dynamic cache even if static was requested.
    used_static = bool(use_static_cache and not used_mtp)

    try:
        with torch.inference_mode():
            out = model.generate(**gen_kwargs)
    except Exception as e:
        msg = str(e).lower()
        # Một số backend/model chưa nhận static cache hoặc assistant + static cùng lúc.
        if "cache_implementation" in msg or "static" in msg:
            print("Static cache lỗi -> fallback dynamic cache:", repr(e))
            gen_kwargs.pop("cache_implementation", None)
            used_static = False
            with torch.inference_mode():
                out = model.generate(**gen_kwargs)
        elif used_mtp:
            print("MTP lỗi -> fallback baseline không assistant:", repr(e))
            gen_kwargs.pop("assistant_model", None)
            used_mtp = False
            with torch.inference_mode():
                out = model.generate(**gen_kwargs)
        else:
            raise

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t1 = time.perf_counter()

    output_tokens = int(out[0].shape[-1] - input_tokens)
    elapsed = t1 - t0
    tok_s = output_tokens / elapsed if elapsed > 0 else 0.0
    decoded = processor.decode(out[0][input_tokens:], skip_special_tokens=True)

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "seconds": elapsed,
        "tok_s": tok_s,
        "used_static_cache": used_static,
        "used_mtp": used_mtp,
        "mtp_tokens": actual_mtp_tokens,
        "mtp_schedule": actual_mtp_schedule,
        "mtp_threshold": actual_mtp_threshold,
        "text": decoded,
    }

def bench(
    name,
    prompt,
    max_new_tokens=160,
    use_mtp=False,
    parse=False,
    mtp_tokens=None,
    mtp_schedule=None,
    mtp_threshold=None,
):
    print("\n" + "-" * 110)
    print("TEST:", name)
    print("-" * 110)

    inp = make_inputs(prompt)
    print("active_target:", active_target_id)
    print("target_mode:", target_mode)
    print("active_assistant:", active_assistant_id if assistant_model is not None else None)
    print("assistant_mode:", assistant_mode)
    print("prompt_chars:", len(prompt))
    print("prompt_tokens:", int(inp["input_ids"].shape[-1]))
    print("max_new_tokens:", max_new_tokens)
    print("use_mtp_requested:", use_mtp)
    if use_mtp and assistant_model is not None:
        configure_mtp_assistant(assistant_model, mtp_tokens, mtp_schedule, mtp_threshold)
        cfg = getattr(assistant_model, "generation_config", None)
        if cfg is not None:
            print("mtp_num_assistant_tokens:", getattr(cfg, "num_assistant_tokens", None))
            print("mtp_schedule:", getattr(cfg, "num_assistant_tokens_schedule", None))
            print("mtp_threshold:", getattr(cfg, "assistant_confidence_threshold", None))

    r = generate_once(
        prompt,
        max_new_tokens=max_new_tokens,
        use_mtp=use_mtp,
        use_static_cache=True,
        mtp_tokens=mtp_tokens,
        mtp_schedule=mtp_schedule,
        mtp_threshold=mtp_threshold,
    )

    print("input_tokens:", r["input_tokens"])
    print("output_tokens:", r["output_tokens"])
    print("seconds:", round(r["seconds"], 3))
    print("tokens_per_second:", round(r["tok_s"], 3))
    print("used_static_cache:", r["used_static_cache"])
    print("used_mtp:", r["used_mtp"])
    if r["used_mtp"]:
        print("actual_mtp_tokens:", r["mtp_tokens"])
        print("actual_mtp_schedule:", r["mtp_schedule"])
        print("actual_mtp_threshold:", r["mtp_threshold"])
    print("preview:")
    print(r["text"][:1600])

    if parse:
        try:
            obj = parse_json_loose(r["text"])
            print("\nJSON PARSE: OK")
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:2500])
        except Exception as e:
            print("\nJSON PARSE: LỖI:", repr(e))

    return r

# ------------------------------------------------------------
# 9) CHẠY BENCHMARK
# ------------------------------------------------------------
print("=" * 110)
print("6) BENCHMARK FAST MTP / BASELINE OPTIONAL")
print("=" * 110)

results = []

fast_use_mtp = assistant_model is not None

if RUN_WARMUP:
    warmup_name = "warmup_mtp_short" if fast_use_mtp else "warmup_baseline_short"
    results.append((warmup_name, bench(
        warmup_name,
        SHORT_PROMPT,
        max_new_tokens=48,
        use_mtp=fast_use_mtp,
        parse=False,
    )))

if RUN_BASELINE_COMPARE:
    results.append(("baseline_dfuzz", bench(
        "baseline_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=False,
        parse=True,
    )))
else:
    print("\nFAST MODE: bỏ qua baseline_dfuzz. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.")

if RUN_FULL_DFUZZ and assistant_model is not None and RUN_DFUZZ_MTP_SWEEP:
    seen_mtp_configs = set()
    for cfg in MTP_DFUZZ_SWEEP_CONFIGS:
        key = (cfg["schedule"], cfg["tokens"], cfg["threshold"])
        if key in seen_mtp_configs:
            continue
        seen_mtp_configs.add(key)
        name = f"mtp_dfuzz_{cfg['schedule']}_n{cfg['tokens']}_t{cfg['threshold']:g}"
        results.append((name, bench(
            name,
            DFUZZ_PROMPT,
            max_new_tokens=180,
            use_mtp=True,
            parse=True,
            mtp_tokens=cfg["tokens"],
            mtp_schedule=cfg["schedule"],
            mtp_threshold=cfg["threshold"],
        )))
elif RUN_FULL_DFUZZ and assistant_model is not None:
    results.append(("mtp_dfuzz", bench(
        "mtp_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=True,
        parse=True,
    )))
elif RUN_FULL_DFUZZ and not RUN_BASELINE_COMPARE:
    results.append(("baseline_dfuzz", bench(
        "baseline_dfuzz",
        DFUZZ_PROMPT,
        max_new_tokens=180,
        use_mtp=False,
        parse=True,
    )))
elif RUN_FULL_DFUZZ:
    print("\nKhông có assistant_model -> bỏ qua MTP benchmark.")

# Một lượt output ngắn hơn, vì DFUZZ thực tế nên ép JSON ngắn.
SHORT_JSON_DFUZZ_PROMPT = """
Rút ca biên từ check C++ sau. Chỉ trả JSON một dòng.
Schema: {"api":"str","bad":"str","p":["str"],"t":["str"],"conf":0.0}
Signature: upsample_bicubic2d(Tensor input, List output_size, Bool align_corners, Optional scales_h, Optional scales_w)
Check: input.numel()!=0 || product(input.sizes()[1:]) != 0
""".strip()

if RUN_BASELINE_COMPARE:
    results.append(("baseline_short_json", bench(
        "baseline_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=False,
        parse=True,
    )))
else:
    print("\nFAST MODE: bỏ qua baseline_short_json. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.")

if RUN_SHORT_JSON and assistant_model is not None and RUN_MTP_SWEEP:
    seen_mtp_tokens = set()
    for n in MTP_SWEEP_TOKENS:
        if n in seen_mtp_tokens:
            continue
        seen_mtp_tokens.add(n)
        name = f"mtp_short_json_n{n}"
        results.append((name, bench(
            name,
            SHORT_JSON_DFUZZ_PROMPT,
            max_new_tokens=96,
            use_mtp=True,
            parse=True,
            mtp_tokens=n,
        )))
elif RUN_SHORT_JSON and assistant_model is not None:
    results.append(("mtp_short_json", bench(
        "mtp_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=True,
        parse=True,
    )))
elif RUN_SHORT_JSON and not RUN_BASELINE_COMPARE:
    results.append(("baseline_short_json", bench(
        "baseline_short_json",
        SHORT_JSON_DFUZZ_PROMPT,
        max_new_tokens=96,
        use_mtp=False,
        parse=True,
    )))

# ------------------------------------------------------------
# 10) TỔNG KẾT
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("7) TỔNG KẾT")
print("=" * 110)

summary = []
for name, r in results:
    summary.append({
        "name": name,
        "in_tok": r["input_tokens"],
        "out_tok": r["output_tokens"],
        "seconds": round(r["seconds"], 3),
        "tok_s": round(r["tok_s"], 3),
        "static": r["used_static_cache"],
        "mtp": r["used_mtp"],
        "mtp_n": r.get("mtp_tokens"),
        "mtp_schedule": r.get("mtp_schedule"),
        "mtp_threshold": r.get("mtp_threshold"),
    })

print(json.dumps(summary, ensure_ascii=False, indent=2))

base = next((r for n, r in results if n == "baseline_dfuzz"), None)
mtp_candidates = [(n, r) for n, r in results if n.startswith("mtp_dfuzz")]
mtp_name, mtp = max(mtp_candidates, key=lambda x: x[1]["tok_s"]) if mtp_candidates else (None, None)

if base and mtp:
    speedup = mtp["tok_s"] / base["tok_s"] if base["tok_s"] > 0 else 0
    time_ratio = base["seconds"] / mtp["seconds"] if mtp["seconds"] > 0 else 0
    print("\nSo sánh DFUZZ:")
    print("baseline tok/s:", round(base["tok_s"], 3))
    print("MTP tốt nhất:", mtp_name)
    print("MTP tok/s:", round(mtp["tok_s"], 3))
    print("tok/s speedup:", round(speedup, 3), "x")
    print("wall-time speedup:", round(time_ratio, 3), "x")

meaningful_summary = [x for x in summary if not x["name"].startswith("warmup")]
if meaningful_summary:
    fastest_tok = max(meaningful_summary, key=lambda x: x["tok_s"])
    fastest_wall = min(meaningful_summary, key=lambda x: x["seconds"])
    print("\nNhanh nhất:")
    print("tok/s:", fastest_tok["name"], fastest_tok["tok_s"], "tok/s")
    print("wall-time:", fastest_wall["name"], fastest_wall["seconds"], "s")

print("\nKết luận đọc nhanh:")
print("- Mặc định fast mode: target hiện tại + assistant_model + MTP strict n=4/constant/0.0.")
print("- RUN_DFUZZ_MTP_SWEEP=True sẽ tự thử các cấu hình MTP đáng nghi nhất cho DFUZZ.")
print("- Muốn so baseline lại: đặt RUN_BASELINE_COMPARE=True.")
print("- Nếu MTP OOM/lỗi: script tự rơi về baseline cho test cần thiết.")
print("- Nếu active_target là E4B: đây là hướng mạnh nhất còn có cửa trên T4.")
print("- Nếu fallback sang E2B: E4B không fit môi trường, dùng E2B lọc hàng loạt rồi E4B/A100 kiểm lại case khó.")
print("- Nếu tok/s vẫn thấp dù VRAM thừa: cổ chai là T4 + decoding/quant kernel, không phải thiếu VRAM.")
print_gpu("GPU CUỐI CELL")


0) CÀI / CẬP NHẬT THƯ VIỆN
Bỏ qua pip install vì dependency đã đủ hoặc INSTALL_DEPS=0.
1) GPU TRƯỚC KHI LOAD
Thu May 21 14:41:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                              

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


3) CẤU HÌNH
Target ưu tiên: google/gemma-4-E4B-it
Assistant ưu tiên: google/gemma-4-E4B-it-assistant
Fallback: google/gemma-4-E2B-it
dtype: torch.float16
HF_XET_HIGH_PERFORMANCE: 0
HF_XET_NUM_CONCURRENT_RANGE_GETS: 4
HF_XET_CHUNK_CACHE_SIZE_BYTES: 0
HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY: 1
HF_HUB_DISABLE_XET: None
HF_HOME: /content/hf_cache
TARGET_INT8_THRESHOLD: 0.0
MTP num_assistant_tokens: 4
MTP schedule: constant
MTP threshold: 0.0
RUN_DFUZZ_MTP_SWEEP: True
MTP_DFUZZ_SWEEP_CONFIGS: [{'schedule': 'constant', 'tokens': 2, 'threshold': 0.0}, {'schedule': 'constant', 'tokens': 3, 'threshold': 0.0}, {'schedule': 'constant', 'tokens': 4, 'threshold': 0.0}, {'schedule': 'constant', 'tokens': 6, 'threshold': 0.0}, {'schedule': 'constant', 'tokens': 8, 'threshold': 0.0}, {'schedule': 'heuristic_transient', 'tokens': 4, 'threshold': 0.2}, {'schedule': 'heuristic_transient', 'tokens': 8, 'threshold': 0.2}]
RUN_MTP_SWEEP: True
MTP_SWEEP_TOKENS: [2, 3, 4, 6, 8, 12]
RUN_BASELINE_COMPARE: False


4) LOAD TARGET MODEL
Thử target 8-bit: google/gemma-4-E4B-it


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

OK: assistant 4-bit
OK: MTP strict n=4 schedule=constant threshold=0.0

assistant device map:
  cuda:0: 0.073B params
  total visible: 0.073B params
  OK: không thấy parameter trên CPU.
GPU SAU KHI LOAD ASSISTANT
Thu May 21 14:47:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0  

[transformers] An assistant model is provided, using a dynamic cache instead of a cache of type='static'.


input_tokens: 59
output_tokens: 12
seconds: 2.157
tokens_per_second: 5.564
used_static_cache: False
used_mtp: True
actual_mtp_tokens: 4
actual_mtp_schedule: constant
actual_mtp_threshold: 0.0
preview:
{"ok":true,"edge":"Tensor rỗng"}

FAST MODE: bỏ qua baseline_dfuzz. Đặt RUN_BASELINE_COMPARE=True nếu cần so lại.

--------------------------------------------------------------------------------------------------------------
TEST: mtp_dfuzz_constant_n2_t0
--------------------------------------------------------------------------------------------------------------
active_target: google/gemma-4-E4B-it
target_mode: 8bit
active_assistant: google/gemma-4-E4B-it-assistant
assistant_mode: 4bit_nf4
prompt_chars: 800
prompt_tokens: 319
max_new_tokens: 180
use_mtp_requested: True
mtp_num_assistant_tokens: 2
mtp_schedule: constant
mtp_threshold: 0.0
input_tokens: 319
output_tokens: 173
seconds: 13.843
tokens_per_second: 12.498
used_static_cache: False
used_mtp: True
actual_mtp_tokens: 2
actual_mtp

In [ ]:
# ============================================================
# GEMMA 4 E4B + ASSISTANT CACHE BACKUP FOR COLAB
#
# Default action:
#   1. Download target + assistant into /content/hf_cache.
#   2. Archive /content/hf_cache to one large tar file on Google Drive.
#
# Why:
#   - /content is fastest for model loading/running in Colab.
#   - Drive is only used as persistent storage between runtimes.
#   - One tar file is much friendlier to Drive than many cache files.
#
# Usage in Colab:
#   python cache_gemma4_to_drive.py
#
# Restore in a new Colab runtime:
#   CACHE_ACTION=restore python cache_gemma4_to_drive.py
# ============================================================

import gc
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path


# -----------------------------
# Safe defaults for Colab free/T4
# -----------------------------
if os.path.exists("/content"):
    os.environ.setdefault("HF_HOME", "/content/hf_cache")
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# High-performance Xet can use more CPU/RAM. Keep it safe by default.
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "0")
os.environ.setdefault("HF_XET_NUM_CONCURRENT_RANGE_GETS", "4")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
os.environ.setdefault("HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY", "1")


PIP_PACKAGES = [
    "huggingface_hub>=0.36.0",
    "hf_xet>=1.2.0",
]

TARGET_MODEL_ID = os.environ.get("TARGET_MODEL_ID", "google/gemma-4-E4B-it")
ASSISTANT_MODEL_ID = os.environ.get("ASSISTANT_MODEL_ID", "google/gemma-4-E4B-it-assistant")
MODEL_IDS = [x.strip() for x in os.environ.get(
    "MODEL_IDS",
    f"{TARGET_MODEL_ID},{ASSISTANT_MODEL_ID}",
).split(",") if x.strip()]

HF_HOME = Path(os.environ.get("HF_HOME", "/content/hf_cache")).expanduser()
DRIVE_ROOT = Path(os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive")).expanduser()
DRIVE_ARCHIVE_DIR = Path(os.environ.get(
    "DRIVE_ARCHIVE_DIR",
    str(DRIVE_ROOT / "hf_model_archives"),
)).expanduser()
ARCHIVE_NAME = os.environ.get("ARCHIVE_NAME", "gemma4_e4b_mtp_hf_cache.tar")
ARCHIVE_PATH = DRIVE_ARCHIVE_DIR / ARCHIVE_NAME

# download_export | download | export | restore
CACHE_ACTION = os.environ.get("CACHE_ACTION", "download_export").strip().lower()
USE_ZSTD = os.environ.get("USE_ZSTD", "0").strip().lower() in {"1", "true", "yes"}
SNAPSHOT_MAX_WORKERS = int(os.environ.get("SNAPSHOT_MAX_WORKERS", "4"))
ALLOW_PATTERNS = [
    "*.safetensors",
    "*.json",
    "*.model",
    "*.txt",
    "*.py",
    "*.jinja",
    "tokenizer*",
    "spiece*",
]


def version_tuple(text):
    nums = [int(x) for x in re.findall(r"\d+", text)[:3]]
    return tuple((nums + [0, 0, 0])[:3])


def installed_between(pkg, min_version=None):
    try:
        current = version(pkg)
    except PackageNotFoundError:
        return False
    if min_version is None:
        return True
    return version_tuple(current) >= version_tuple(min_version)


def should_install_deps():
    mode = os.environ.get("INSTALL_DEPS", "auto").strip().lower()
    if mode in {"0", "false", "no", "skip"}:
        return False
    if mode in {"1", "true", "yes", "force"}:
        return True
    return not (
        installed_between("huggingface_hub", "0.36.0")
        and installed_between("hf_xet", "1.2.0")
    )


def run_cmd(cmd, check=True):
    print("\n$", " ".join(str(x) for x in cmd) if isinstance(cmd, list) else cmd)
    t0 = time.perf_counter()
    p = subprocess.run(
        cmd,
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    dt = time.perf_counter() - t0
    if p.stdout:
        print(p.stdout)
    print(f"exit={p.returncode} seconds={dt:.3f}")
    if check and p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd, output=p.stdout)
    return p.stdout


def print_system_state(title):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)
    for cmd in [
        "nvidia-smi",
        "free -h",
        f"df -h /content {HF_HOME} {DRIVE_ROOT} 2>/dev/null",
    ]:
        try:
            print(run_cmd(cmd, check=False))
        except Exception as e:
            print("state command failed:", repr(e))


def mount_drive_if_needed():
    if not os.path.exists("/content"):
        raise RuntimeError("This script is intended for Colab when using Drive export/restore.")
    if DRIVE_ROOT.exists():
        return
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not DRIVE_ROOT.exists():
        raise RuntimeError(f"Drive mount not found: {DRIVE_ROOT}")


def login_if_possible():
    from huggingface_hub import notebook_login

    hf_token = os.environ.get("HF_TOKEN")
    try:
        from google.colab import userdata
        hf_token = hf_token or userdata.get("HF_TOKEN")
    except Exception:
        pass

    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN found. Downloads will use it without archiving the token file.")
    else:
        print("HF_TOKEN not found. Public/rate-limited downloads may still work.")
        try:
            notebook_login()
        except Exception as e:
            print("notebook_login skipped/failed:", repr(e))


def download_models_to_cache():
    from huggingface_hub import snapshot_download

    HF_HOME.mkdir(parents=True, exist_ok=True)
    rows = []
    for repo_id in MODEL_IDS:
        print("\n" + "-" * 110)
        print("DOWNLOAD:", repo_id)
        print("-" * 110)
        t0 = time.perf_counter()
        path = snapshot_download(
            repo_id=repo_id,
            cache_dir=str(HF_HOME / "hub"),
            allow_patterns=ALLOW_PATTERNS,
            max_workers=SNAPSHOT_MAX_WORKERS,
            token=os.environ.get("HF_TOKEN") or None,
        )
        seconds = time.perf_counter() - t0
        row = {
            "repo": repo_id,
            "snapshot_path": path,
            "seconds": round(seconds, 3),
        }
        rows.append(row)
        print(json.dumps(row, ensure_ascii=False, indent=2))
    return rows


def sha256_file(path, chunk_mb=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_mb * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def write_manifest(archive_path, rows, archive_members=None):
    manifest = {
        "archive": str(archive_path),
        "archive_size_bytes": archive_path.stat().st_size if archive_path.exists() else None,
        "sha256": sha256_file(archive_path) if archive_path.exists() else None,
        "hf_home": str(HF_HOME),
        "archive_members": archive_members or [],
        "auth_files_archived": False,
        "model_ids": MODEL_IDS,
        "download_rows": rows,
        "env": {
            "HF_XET_HIGH_PERFORMANCE": os.environ.get("HF_XET_HIGH_PERFORMANCE"),
            "HF_XET_NUM_CONCURRENT_RANGE_GETS": os.environ.get("HF_XET_NUM_CONCURRENT_RANGE_GETS"),
            "HF_XET_CHUNK_CACHE_SIZE_BYTES": os.environ.get("HF_XET_CHUNK_CACHE_SIZE_BYTES"),
            "HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY": os.environ.get("HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY"),
            "HF_HOME": os.environ.get("HF_HOME"),
        },
    }
    manifest_path = archive_path.with_suffix(archive_path.suffix + ".manifest.json")
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Wrote manifest:", manifest_path)
    return manifest


def cache_archive_members():
    # Archive cache data only, not HF_HOME/token or other auth files.
    members = []
    for name in ["hub", "xet"]:
        path = HF_HOME / name
        if path.exists():
            members.append(str(path.relative_to(HF_HOME.parent)))
    if not members:
        raise RuntimeError(f"No cache data found under {HF_HOME}; expected hub/ and optionally xet/.")
    return members


def export_cache_to_drive(download_rows=None):
    mount_drive_if_needed()
    DRIVE_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

    if not HF_HOME.exists():
        raise RuntimeError(f"HF_HOME does not exist: {HF_HOME}")

    archive_path = ARCHIVE_PATH
    if USE_ZSTD:
        if shutil.which("zstd") is None:
            print("USE_ZSTD=1 but zstd command is missing; falling back to plain tar.")
        elif not archive_path.name.endswith(".tar.zst"):
            archive_path = archive_path.with_suffix(archive_path.suffix + ".zst")

    tmp_path = archive_path.with_name(archive_path.name + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    print_system_state("BEFORE EXPORT")
    t0 = time.perf_counter()
    parent = HF_HOME.parent
    members = cache_archive_members()
    member_args = " ".join(shlex_quote(x) for x in members)
    print("Archive members:", members)
    print("Auth files such as HF_HOME/token are intentionally not archived.")

    if USE_ZSTD and shutil.which("zstd") is not None:
        cmd = f"tar -C {shlex_quote(str(parent))} -I 'zstd -1 -T1' -cf {shlex_quote(str(tmp_path))} {member_args}"
    else:
        cmd = f"tar -C {shlex_quote(str(parent))} -cf {shlex_quote(str(tmp_path))} {member_args}"
    run_cmd(cmd)
    os.replace(tmp_path, archive_path)
    seconds = time.perf_counter() - t0

    print(f"EXPORT OK: {archive_path}")
    print(f"archive_gb={archive_path.stat().st_size / (1024 ** 3):.3f} seconds={seconds:.3f}")
    manifest = write_manifest(archive_path, download_rows or [], members)
    print(json.dumps(manifest, ensure_ascii=False, indent=2)[:4000])
    print_system_state("AFTER EXPORT")


def restore_cache_from_drive():
    mount_drive_if_needed()
    archive_path = ARCHIVE_PATH
    if not archive_path.exists() and archive_path.with_suffix(archive_path.suffix + ".zst").exists():
        archive_path = archive_path.with_suffix(archive_path.suffix + ".zst")
    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    print_system_state("BEFORE RESTORE")
    HF_HOME.parent.mkdir(parents=True, exist_ok=True)

    t0 = time.perf_counter()
    if str(archive_path).endswith(".tar.zst"):
        cmd = f"tar -C {shlex_quote(str(HF_HOME.parent))} -I zstd -xf {shlex_quote(str(archive_path))}"
    else:
        cmd = f"tar -C {shlex_quote(str(HF_HOME.parent))} -xf {shlex_quote(str(archive_path))}"
    run_cmd(cmd)
    seconds = time.perf_counter() - t0

    print(f"RESTORE OK: {archive_path} -> {HF_HOME}")
    print(f"seconds={seconds:.3f}")
    print("Tip: after restore, set HF_HUB_OFFLINE=1 if you want to force cached-only loading.")
    print_system_state("AFTER RESTORE")


def shlex_quote(text):
    # Small local quote helper to avoid importing shlex before old notebook cells finish dependency setup.
    return "'" + text.replace("'", "'\"'\"'") + "'"


def main():
    print_system_state("START")
    print(json.dumps({
        "CACHE_ACTION": CACHE_ACTION,
        "HF_HOME": str(HF_HOME),
        "DRIVE_ARCHIVE_DIR": str(DRIVE_ARCHIVE_DIR),
        "ARCHIVE_PATH": str(ARCHIVE_PATH),
        "MODEL_IDS": MODEL_IDS,
        "SNAPSHOT_MAX_WORKERS": SNAPSHOT_MAX_WORKERS,
        "USE_ZSTD": USE_ZSTD,
    }, ensure_ascii=False, indent=2))

    if should_install_deps():
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-U", *PIP_PACKAGES])
    else:
        print("Dependencies already OK or INSTALL_DEPS=0.")

    login_if_possible()

    rows = []
    if CACHE_ACTION in {"download", "download_export"}:
        rows = download_models_to_cache()
        gc.collect()

    if CACHE_ACTION in {"export", "download_export"}:
        export_cache_to_drive(rows)
    elif CACHE_ACTION == "restore":
        restore_cache_from_drive()
    elif CACHE_ACTION == "download":
        print("DOWNLOAD OK. No Drive export because CACHE_ACTION=download.")
    else:
        raise ValueError("CACHE_ACTION must be one of: download_export, download, export, restore")


if __name__ == "__main__":
    main()



START

$ nvidia-smi
Thu May 21 15:09:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



--------------------------------------------------------------------------------------------------------------
DOWNLOAD: google/gemma-4-E4B-it
--------------------------------------------------------------------------------------------------------------


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

{
  "repo": "google/gemma-4-E4B-it",
  "snapshot_path": "/content/hf_cache/hub/models--google--gemma-4-E4B-it/snapshots/d6436b3d62967e1af08bbb046c6300b2a9ae8e85",
  "seconds": 177.237
}

--------------------------------------------------------------------------------------------------------------
DOWNLOAD: google/gemma-4-E4B-it-assistant
--------------------------------------------------------------------------------------------------------------


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

{
  "repo": "google/gemma-4-E4B-it-assistant",
  "snapshot_path": "/content/hf_cache/hub/models--google--gemma-4-E4B-it-assistant/snapshots/4a5c666f89be588c72e0b3a9b49c118513cedff6",
  "seconds": 4.574
}
Mounting Google Drive...
Mounted at /content/drive

BEFORE EXPORT

$ nvidia-smi
Thu May 21 15:12:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Of

In [ ]:
# ============================================================
# QUICK RESTORE SPEED BENCHMARK FOR DRIVE CACHE ARCHIVE
#
# Measures two restore strategies for the Gemma HF cache archive:
#   1. direct_extract: tar reads archive directly from Drive and extracts to /content.
#   2. copy_then_extract: copy archive to /content first, then extract from local disk.
#
# Default is safe:
#   - extracts into /content/cache_restore_bench, not /content/hf_cache.
#   - deletes only /content/cache_restore_bench and optional local temp archive.
#
# Usage in Colab:
#   python bench_drive_cache_restore.py
#
# Useful env:
#   ARCHIVE_PATH=/content/drive/MyDrive/hf_model_archives/gemma4_e4b_mtp_hf_cache.tar
#   BENCH_MODE=both                  # both | direct_extract | copy_then_extract
#   CLEANUP_AFTER=1                  # delete extracted test data after each run
# ============================================================

import json
import os
import shutil
import subprocess
import time
from pathlib import Path


DRIVE_ROOT = Path(os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive")).expanduser()
ARCHIVE_PATH = Path(os.environ.get(
    "ARCHIVE_PATH",
    str(DRIVE_ROOT / "hf_model_archives" / "gemma4_e4b_mtp_hf_cache.tar"),
)).expanduser()
BENCH_ROOT = Path(os.environ.get("BENCH_ROOT", "/content/cache_restore_bench")).expanduser()
LOCAL_ARCHIVE = Path(os.environ.get("LOCAL_ARCHIVE", "/content/gemma4_e4b_mtp_hf_cache.tar")).expanduser()
BENCH_MODE = os.environ.get("BENCH_MODE", "both").strip().lower()
CLEANUP_AFTER = os.environ.get("CLEANUP_AFTER", "1").strip().lower() not in {"0", "false", "no"}


def q(path):
    return "'" + str(path).replace("'", "'\"'\"'") + "'"


def run_cmd(cmd, check=True):
    print("\n$", cmd)
    t0 = time.perf_counter()
    p = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    seconds = time.perf_counter() - t0
    if p.stdout:
        print(p.stdout)
    print(f"exit={p.returncode} seconds={seconds:.3f}")
    if check and p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd, output=p.stdout)
    return seconds, p.stdout


def mount_drive_if_needed():
    if not os.path.exists("/content"):
        raise RuntimeError("This script is intended for Colab.")
    if DRIVE_ROOT.exists():
        return
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not DRIVE_ROOT.exists():
        raise RuntimeError(f"Drive mount not found: {DRIVE_ROOT}")


def safe_rmtree(path):
    path = Path(path).resolve()
    allowed = Path("/content/cache_restore_bench").resolve()
    if path == allowed or allowed in path.parents:
        shutil.rmtree(path, ignore_errors=True)
        return
    raise RuntimeError(f"Refusing to delete outside benchmark root: {path}")


def print_state(title):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)
    for cmd in [
        "free -h",
        f"df -h /content {DRIVE_ROOT} 2>/dev/null",
        f"du -h {q(ARCHIVE_PATH)} 2>/dev/null",
    ]:
        run_cmd(cmd, check=False)


def archive_size_gb(path):
    return path.stat().st_size / (1024 ** 3)


def extracted_size_gb(path):
    seconds, out = run_cmd(f"du -sb {q(path)} 2>/dev/null | awk '{{print $1}}'", check=False)
    try:
        return int(out.strip().splitlines()[-1]) / (1024 ** 3)
    except Exception:
        return None


def benchmark_direct_extract():
    dest = BENCH_ROOT / "direct"
    safe_rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)

    print_state("BEFORE DIRECT EXTRACT")
    cmd = f"tar -C {q(dest)} -xf {q(ARCHIVE_PATH)}"
    seconds, _ = run_cmd(cmd)
    size_gb = extracted_size_gb(dest)

    row = {
        "mode": "direct_extract",
        "archive_gb": round(archive_size_gb(ARCHIVE_PATH), 3),
        "extracted_gb": round(size_gb, 3) if size_gb is not None else None,
        "seconds": round(seconds, 3),
        "archive_read_gb_s": round(archive_size_gb(ARCHIVE_PATH) / seconds, 3) if seconds > 0 else None,
    }
    print(json.dumps(row, ensure_ascii=False, indent=2))
    print_state("AFTER DIRECT EXTRACT")

    if CLEANUP_AFTER:
        safe_rmtree(dest)
    return row


def benchmark_copy_then_extract():
    dest = BENCH_ROOT / "copy_then_extract"
    safe_rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    if LOCAL_ARCHIVE.exists():
        LOCAL_ARCHIVE.unlink()

    print_state("BEFORE COPY THEN EXTRACT")
    copy_seconds, _ = run_cmd(f"cp {q(ARCHIVE_PATH)} {q(LOCAL_ARCHIVE)}")
    extract_seconds, _ = run_cmd(f"tar -C {q(dest)} -xf {q(LOCAL_ARCHIVE)}")
    total_seconds = copy_seconds + extract_seconds
    size_gb = extracted_size_gb(dest)

    row = {
        "mode": "copy_then_extract",
        "archive_gb": round(archive_size_gb(ARCHIVE_PATH), 3),
        "extracted_gb": round(size_gb, 3) if size_gb is not None else None,
        "copy_seconds": round(copy_seconds, 3),
        "extract_seconds": round(extract_seconds, 3),
        "total_seconds": round(total_seconds, 3),
        "copy_gb_s": round(archive_size_gb(ARCHIVE_PATH) / copy_seconds, 3) if copy_seconds > 0 else None,
        "total_gb_s": round(archive_size_gb(ARCHIVE_PATH) / total_seconds, 3) if total_seconds > 0 else None,
    }
    print(json.dumps(row, ensure_ascii=False, indent=2))
    print_state("AFTER COPY THEN EXTRACT")

    if CLEANUP_AFTER:
        safe_rmtree(dest)
        if LOCAL_ARCHIVE.exists():
            LOCAL_ARCHIVE.unlink()
    return row


def main():
    mount_drive_if_needed()
    if not ARCHIVE_PATH.exists():
        raise FileNotFoundError(f"Archive not found: {ARCHIVE_PATH}")
    BENCH_ROOT.mkdir(parents=True, exist_ok=True)

    print(json.dumps({
        "ARCHIVE_PATH": str(ARCHIVE_PATH),
        "BENCH_ROOT": str(BENCH_ROOT),
        "LOCAL_ARCHIVE": str(LOCAL_ARCHIVE),
        "BENCH_MODE": BENCH_MODE,
        "CLEANUP_AFTER": CLEANUP_AFTER,
    }, ensure_ascii=False, indent=2))

    rows = []
    if BENCH_MODE in {"both", "direct_extract"}:
        rows.append(benchmark_direct_extract())
    if BENCH_MODE in {"both", "copy_then_extract"}:
        rows.append(benchmark_copy_then_extract())
    if BENCH_MODE not in {"both", "direct_extract", "copy_then_extract"}:
        raise ValueError("BENCH_MODE must be one of: both, direct_extract, copy_then_extract")

    print("\n" + "=" * 110)
    print("SUMMARY")
    print("=" * 110)
    print(json.dumps(rows, ensure_ascii=False, indent=2))

    if rows:
        winner = min(rows, key=lambda r: r.get("total_seconds", r.get("seconds", 1e99)))
        print("FASTEST:", winner["mode"])


if __name__ == "__main__":
    main()


Mounted at /content/drive
{
  "ARCHIVE_PATH": "/content/drive/MyDrive/hf_model_archives/gemma4_e4b_mtp_hf_cache.tar",
  "BENCH_ROOT": "/content/cache_restore_bench",
  "LOCAL_ARCHIVE": "/content/gemma4_e4b_mtp_hf_cache.tar",
  "BENCH_MODE": "both",
  "CLEANUP_AFTER": true
}

BEFORE DIRECT EXTRACT

$ free -h
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.4Gi       6.5Gi       2.0Mi       3.8Gi        10Gi
Swap:             0B          0B          0B

exit=0 seconds=0.007

$ df -h /content /content/drive/MyDrive 2>/dev/null
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   70G  39% /
drive           113G   47G   67G  42% /content/drive

exit=0 seconds=0.015

$ du -h '/content/drive/MyDrive/hf_model_archives/gemma4_e4b_mtp_hf_cache.tar' 2>/dev/null
16G	/content/drive/MyDrive/hf_model_archives/gemma4_e4b_mtp_hf_cache.tar

exit=0 seconds=0.007

$ tar -C '/content/cache_restore_bench/direct' -xf '/cont